# Classical residual-heterogeneity diagnostics for the 1D causal ANM

This notebook adds three transparent classical diagnostic families to the
*existing* `toy_additive_noise_gof_1d` causal-arrow experiment:

1. finite-dimensional BP/White-style variance-score tests;
2. binned Levene and Brown--Forsythe dispersion tests;
3. finite-feature residual quantile-regression tests;
4. permutation-calibrated finite-feature HSIC tests using the same squared-residual
   signal as the BP/White-style variance scores.

Goldfeld--Quandt is deliberately excluded.

## Exact experiment reused

The repository calls the two directions `Y on X` and `X on Y`; below these
are reported as \(X\to Y\) and \(Y\to X\), respectively.  The four
conditions are \(\rho\in\{0,0.35\}\) crossed with both directions.  Only
\(\rho=0,\ X\to Y\) is a true additive-noise null.

The authoritative current experiment uses:

- \(N=250\) and 100 paired Monte Carlo repetitions;
- the exact DGP implemented in cell `e5b540f5` of
  `toy_additive_noise_gof_1d.ipynb`;
- Gaussian `sklearn` kernel ridge regression, tuned separately in every
  first-stage fit by deterministic five-fold `GridSearchCV` with MSE loss,
  `alpha` in `{1e-3, 1e-2, 1e-1, 1, 10}`, and RBF length scale in
  `{0.5, 1, 2, 4, 8}`;
- five deterministic contiguous outer folds of 50 observations.  Cross-fit
  residuals train on the other 200 observations.  Ordinary splits use the
  existing nested prefix-fold convention: 20/40/60/80% train and
  80/60/40/20% evaluate;
- data seed `82345 + 1000 * repetition + 100 * rho_index`, with one dataset
  generated per `(repetition, rho)` and reused for both directions and every
  diagnostic.

The prefix splits are not newly shuffled: they exactly follow the current
notebook.  Since simulated rows are i.i.d. and randomly ordered, they are
random in distribution.  Inner CV is also unshuffled.  All second-stage
configurations below reuse one residual vector per
`(dataset, direction, residual scheme)`.

Cross-fitting does not automatically validate any textbook null reference
distribution after a learned first stage.  Results are therefore called
**cross-fitted diagnostic variants**, and their calibration is measured in
the one true-null panel. The new permutation-HSIC procedure is added in
parallel: none of the existing ANOVA/BP-style results is replaced or removed.

For the reported fresh 1,000-repetition table use `run_toy_additive_noise_selected_1000.py` and `summarize_toy_additive_noise_selected_1000.py`. This notebook records the preliminary 100-repetition configuration search; the runner imports only its function-definition cells. See `CAUSAL_COVARIATE_REPRODUCTION.md`.


In [ ]:
from __future__ import annotations

from collections import OrderedDict
from datetime import datetime, timezone
import hashlib
import json
import math
import os
from pathlib import Path
import platform
import time
import warnings

from IPython.display import Markdown, display
import jax.numpy as jnp
import nbformat
import numpy as np
import pandas as pd
import scipy
from scipy import linalg, stats
from scipy.stats import chi2
import sklearn
import statsmodels
from statsmodels.regression.quantile_regression import QuantReg
from statsmodels.tools.sm_exceptions import ConvergenceWarning, IterationLimitWarning

from multivariate_crossfit_core import (
    RunConfig,
    build_fold_masks,
    fit_mean_forecasts,
    fit_mean_on_split,
    validate_config,
)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 180)

ALPHA = 0.05
RESULT_DIR = Path("results/toy_additive_noise_classical_diagnostics")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

LONG_CSV = RESULT_DIR / "classical_diagnostics_long.csv"
VARIANCE_CSV = RESULT_DIR / "aggregate_variance_score.csv"
LEVENE_CSV = RESULT_DIR / "aggregate_levene.csv"
BROWN_FORSYTHE_CSV = RESULT_DIR / "aggregate_brown_forsythe.csv"
QUANTILE_CSV = RESULT_DIR / "aggregate_quantile.csv"
NUMERIC_AGGREGATE_CSV = RESULT_DIR / "aggregate_numeric_long.csv"
PERMUTATION_HSIC_CSV = RESULT_DIR / "aggregate_squared_residual_feature_hsic_permutation.csv"
CALIBRATION_COMPARISON_CSV = RESULT_DIR / "variance_score_calibration_comparison.csv"
ORACLE_PERMUTATION_LONG_CSV = RESULT_DIR / "oracle_squared_residual_feature_hsic_permutation_long.csv"
ORACLE_PERMUTATION_SUMMARY_CSV = RESULT_DIR / "oracle_squared_residual_feature_hsic_permutation_summary.csv"
VERIFICATION_CSV = RESULT_DIR / "verification_checks.csv"
NULL_SMOKE_CSV = RESULT_DIR / "direct_error_null_smoke.csv"
POWER_SMOKE_CSV = RESULT_DIR / "strong_variance_smoke.csv"
METADATA_JSON = RESULT_DIR / "run_metadata.json"
PROGRESS_JSON = RESULT_DIR / "progress.json"
BASELINE_REPETITIONS_CSV = Path(
    "results/toy_additive_noise_gof_1d/"
    "toy_additive_noise_gof_1d_20260502_233746_repetition_records.csv"
)

print(
    {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "scipy": scipy.__version__,
        "pandas": pd.__version__,
        "sklearn": sklearn.__version__,
        "statsmodels": statsmodels.__version__,
    }
)

## Reuse the repository DGP and configuration without copying them

The one-dimensional DGP is notebook-native rather than exported from a
Python module.  To avoid silently drifting from it, the next cell reads and
executes only the existing DGP and configuration source cells.  It does not
execute the original notebook or its HSIC experiment.  SHA-256 hashes make
the precise reused source visible in the saved metadata.

In [ ]:
SOURCE_NOTEBOOK = Path("submission_toy_additive_noise_gof_1d.ipynb")
DGP_CELL_ID = "e5b540f5"
CONFIG_CELL_ID = "6d59aeca"
BASELINE_RUN_JSON = Path(
    "results/toy_additive_noise_gof_1d/"
    "toy_additive_noise_gof_1d_20260502_233746.json"
)


def source_cell(notebook_path: Path, cell_id: str) -> str:
    notebook = nbformat.read(notebook_path, as_version=4)
    matches = [cell.source for cell in notebook.cells if cell.get("id") == cell_id]
    if len(matches) != 1:
        raise RuntimeError(
            f"Expected exactly one cell {cell_id!r} in {notebook_path}, found {len(matches)}."
        )
    return matches[0]


dgp_source = source_cell(SOURCE_NOTEBOOK, DGP_CELL_ID)
dgp_namespace = {"np": np}
exec(compile(dgp_source, f"{SOURCE_NOTEBOOK}#{DGP_CELL_ID}", "exec"), dgp_namespace)

config_source = source_cell(SOURCE_NOTEBOOK, CONFIG_CELL_ID)
config_namespace = {
    **dgp_namespace,
    "RunConfig": RunConfig,
    "validate_config": validate_config,
}
exec(
    compile(config_source, f"{SOURCE_NOTEBOOK}#{CONFIG_CELL_ID}", "exec"),
    config_namespace,
)

simulate_anm = dgp_namespace["simulate_anm"]
direction_arrays = dgp_namespace["direction_arrays"]
f_true = dgp_namespace["f_true"]
RHO_VALUES = tuple(dgp_namespace["RHO_VALUES"])
REPO_DIRECTIONS = tuple(dgp_namespace["DIRECTIONS"])
SEED = int(dgp_namespace["SEED"])
CORE_CONFIG = config_namespace["CORE_CONFIG"]
CORE_N = int(config_namespace["CORE_N"])
CORE_N_REPS = int(config_namespace["CORE_N_REPS"])
N_FOLDS = int(config_namespace["N_FOLDS"])
SPLIT_PREFIXES = tuple(config_namespace["SPLIT_PREFIXES"])
N_PERMUTATIONS = int(CORE_CONFIG.bootstrap_B)

DIRECTION_MAP = {"Y on X": "X->Y", "X on Y": "Y->X"}
DATA_SEED_BASE = SEED + 70_000

assert RHO_VALUES == (0.0, 0.35)
assert REPO_DIRECTIONS == ("Y on X", "X on Y")
assert CORE_N == 250 and CORE_N_REPS == 100 and N_FOLDS == 5
assert SPLIT_PREFIXES == (1, 2, 3, 4)
assert CORE_CONFIG.mean_kernel == "gaussian"
assert tuple(CORE_CONFIG.mean_alphas) == (1e-3, 1e-2, 1e-1, 1.0, 10.0)
assert tuple(CORE_CONFIG.mean_length_scales) == (0.5, 1.0, 2.0, 4.0, 8.0)
assert CORE_CONFIG.mean_cv_folds == 5
assert N_PERMUTATIONS == 399

DGP_SHA256 = hashlib.sha256(dgp_source.encode("utf-8")).hexdigest()
CONFIG_SHA256 = hashlib.sha256(config_source.encode("utf-8")).hexdigest()

experiment_facts = pd.Series(
    {
        "source notebook": str(SOURCE_NOTEBOOK),
        "DGP cell / SHA-256": f"{DGP_CELL_ID} / {DGP_SHA256}",
        "configuration cell / SHA-256": f"{CONFIG_CELL_ID} / {CONFIG_SHA256}",
        "rho values": RHO_VALUES,
        "directions (repository -> report)": DIRECTION_MAP,
        "true null": "rho=0, X->Y only",
        "sample size": CORE_N,
        "Monte Carlo repetitions": CORE_N_REPS,
        "outer folds": N_FOLDS,
        "split training fractions": tuple(p / N_FOLDS for p in SPLIT_PREFIXES),
        "first stage": "Gaussian KRR; deterministic 5-fold CV over 25 configurations",
        "data seed formula": "82345 + 1000*repetition + 100*rho_index",
        "nominal alpha": ALPHA,
        "permutations per finite-feature HSIC test": N_PERMUTATIONS,
        "permutation-count source": "CORE_CONFIG.bootstrap_B (paper baseline)",
    },
    name="value",
)
display(experiment_facts.to_frame())

## Fixed second-stage configurations

For each dataset and direction, the scalar predictor is standardized once:

\[
u_i=(Z_i-\bar Z)/s_Z,
\]

using all predictor values and `ddof=1`, but no response or residual.  The
same `u` is then subset for every split.  Raw feature columns are centered
and scaled on the second-stage predictor values only.  An SVD supplies an
orthogonal basis for exactly the same feature span, removes numerical
dependencies, and records the effective rank \(q\).  This invertible
reparameterization does not alter \(R^2\) or the joint null that all feature
coefficients are zero.

In [ ]:
VARIANCE_BASES = (
    "linear",
    "poly2",
    "poly3",
    "fourier1",
    "fourier2",
    "fourier4",
    "fourier8",
)
QUANTILE_BASES = ("linear", "poly2", "poly3", "fourier2", "fourier4")
HSIC_FEATURE_BASES = VARIANCE_BASES
BIN_COUNTS = (2, 3, 4, 5, 8, 10)
QUANTILE_GRIDS = OrderedDict(
    [
        ("central", (0.25, 0.5, 0.75)),
        ("wide", (0.1, 0.25, 0.5, 0.75, 0.9)),
    ]
)
ALL_QUANTILES = tuple(sorted({tau for grid in QUANTILE_GRIDS.values() for tau in grid}))
MIN_BIN_SIZE = 5
QUANTREG_MAX_ITER = 10_000
QUANTREG_P_TOL = 1e-6


def raw_features(u: np.ndarray, basis: str) -> tuple[np.ndarray, list[str]]:
    u = np.asarray(u, dtype=float).reshape(-1)
    if basis == "linear":
        return u[:, None], ["u"]
    if basis in {"poly2", "poly3"}:
        degree = int(basis[-1])
        return (
            np.column_stack([u**power for power in range(1, degree + 1)]),
            [f"u^{power}" for power in range(1, degree + 1)],
        )
    if basis.startswith("fourier"):
        max_frequency = int(basis.removeprefix("fourier"))
        columns = []
        labels = []
        for frequency in range(1, max_frequency + 1):
            columns.extend([np.sin(frequency * u), np.cos(frequency * u)])
            labels.extend([f"sin({frequency}u)", f"cos({frequency}u)"])
        return np.column_stack(columns), labels
    raise ValueError(f"Unknown feature basis: {basis}")


def prepare_feature_design(u: np.ndarray, basis: str) -> dict:
    """Return a stable orthogonal basis for the requested non-intercept span."""
    raw, labels = raw_features(u, basis)
    if not np.isfinite(raw).all():
        return {"valid": False, "failure_reason": "nonfinite raw features", "q": 0}
    centered = raw - raw.mean(axis=0, keepdims=True)
    scales = centered.std(axis=0, ddof=1)
    raw_scale = np.maximum(1.0, np.max(np.abs(raw), axis=0))
    keep = np.isfinite(scales) & (scales > 1e-12 * raw_scale)
    if not np.any(keep):
        return {"valid": False, "failure_reason": "all feature columns constant", "q": 0}
    scaled = centered[:, keep] / scales[keep]
    U, singular_values, _ = np.linalg.svd(scaled, full_matrices=False)
    if not len(singular_values) or singular_values[0] <= 0:
        return {"valid": False, "failure_reason": "zero feature rank", "q": 0}
    tolerance = singular_values[0] * max(scaled.shape) * np.finfo(float).eps
    q = int(np.sum(singular_values > tolerance))
    if q < 1:
        return {"valid": False, "failure_reason": "zero effective feature rank", "q": 0}
    # Columns have sum of squares n-1, matching ordinary sample standardization.
    features = U[:, :q] * np.sqrt(len(u) - 1)
    design = np.column_stack([np.ones(len(u)), features])
    if np.linalg.matrix_rank(design) != q + 1:
        return {
            "valid": False,
            "failure_reason": "intercept-augmented design is rank deficient",
            "q": q,
        }
    return {
        "valid": True,
        "failure_reason": "",
        "features": features,
        "design": design,
        "q": q,
        "raw_n_features": raw.shape[1],
        "kept_raw_labels": [label for label, flag in zip(labels, keep) if flag],
        "singular_values": singular_values[:q],
    }

## 1. BP/White-style finite variance-score tests

For \(S_i=e_i^2\), explicitly regress \(S\) on an intercept and
\(\Phi(u)\), compute

\[
T=n_{\rm eval}R^2,
\]

and use the Koenker/studentized \(\chi_q^2\) reference distribution.  Here
`poly2` is the one-dimensional White specification.  Fourier bases are
targeted at smooth periodic heterogeneity, but they do not use realized DGP
coefficients or phases and are not oracle tests.

In [ ]:
def variance_score_test(u: np.ndarray, e: np.ndarray, basis: str) -> dict:
    u = np.asarray(u, dtype=float).reshape(-1)
    e = np.asarray(e, dtype=float).reshape(-1)
    n = len(e)
    if len(u) != n or n < 4 or not np.isfinite(u).all() or not np.isfinite(e).all():
        return {"valid": False, "failure_reason": "invalid or nonfinite input"}
    prepared = prepare_feature_design(u, basis)
    if not prepared["valid"]:
        return prepared
    q = prepared["q"]
    if n <= q + 1:
        return {
            "valid": False,
            "failure_reason": "insufficient residual degrees of freedom",
            "q": q,
        }
    S = e**2
    centered_S = S - S.mean()
    sst = float(centered_S @ centered_S)
    if not np.isfinite(sst) or sst <= np.finfo(float).eps * max(1.0, float(S @ S)):
        return {"valid": False, "failure_reason": "degenerate squared residuals", "q": q}
    coef, _, _, _ = np.linalg.lstsq(prepared["design"], S, rcond=None)
    fitted = prepared["design"] @ coef
    sse = float(np.sum((S - fitted) ** 2))
    r_squared = 1.0 - sse / sst
    if r_squared < -1e-10 or r_squared > 1.0 + 1e-10:
        return {
            "valid": False,
            "failure_reason": f"numerically invalid R-squared: {r_squared}",
            "q": q,
        }
    r_squared = float(np.clip(r_squared, 0.0, 1.0))
    statistic = n * r_squared
    p_value = float(chi2.sf(statistic, q))
    valid = np.isfinite(statistic) and np.isfinite(p_value) and 0.0 <= p_value <= 1.0
    return {
        "valid": bool(valid),
        "failure_reason": "" if valid else "nonfinite or out-of-bounds p-value",
        "statistic": statistic,
        "p_value": p_value,
        "df": q,
        "q": q,
        "r_squared": r_squared,
        "raw_n_features": prepared["raw_n_features"],
    }

## 2. Permutation-calibrated finite-feature HSIC on squared residuals

For every variance-score feature map, this parallel procedure tests

\[
H_0:\operatorname{Cov}\{\phi(Z),\varepsilon^2\}=0
\]

with the whitened projection kernel

\[
K=F_c(F_c^\top F_c)^+F_c^\top=QQ^\top,
\qquad T=s_c^\top K s_c,
\qquad s_i=\widehat\varepsilon_i^2.
\]

Here `Q` is the numerically non-degenerate left-singular-vector basis already
used by the analytic BP/White-style test. Thus `T` is left unnormalized and
is exactly the auxiliary regression's explained sum of squares; positive
normalizations would give the same permutation p-value. The residual-side
feature is **exactly the scalar squared residual**. No Gaussian or nonlinear
kernel is applied to it.

For this linear residual feature, the paper's first-order correction is zero
because \(E[2\varepsilon\mid W]=0\). Consequently the first-order debiased
statistic equals the plug-in statistic, and there are deliberately no
separate debiased/non-debiased rows. Cross-fitting remains important for
first-stage overfitting, leaving a second-order first-stage remainder, but it
does not make permutation exact for estimated residuals.

Within each dataset, direction, and residual scheme, one array of 399
permutation indices is reused across all seven feature bases. The value 399
is read from `CORE_CONFIG.bootstrap_B`, matching the executed paper baseline
(the attached specification anticipated 1,000, but the repository baseline
actually uses 399). The first-stage regression is never refit inside a
permutation or separately for this procedure.

In [ ]:
PERMUTATION_SEED_BASE = 2_026_072_401
HSIC_STATISTIC_NORMALIZATION = (
    "none; T = centered squared-residual explained sum of squares"
)


def permutation_seed_for(
    data_seed: int,
    repo_direction: str,
    residual_set: dict,
) -> int:
    direction_code = {"Y on X": 1, "X on Y": 2}[repo_direction]
    if residual_set["residual_scheme"] == "cross_fitted_variant":
        scheme_code = 500
    else:
        scheme_code = int(round(100 * residual_set["training_fraction"]))
    seed_sequence = np.random.SeedSequence(
        [PERMUTATION_SEED_BASE, int(data_seed), direction_code, scheme_code]
    )
    return int(seed_sequence.generate_state(1, dtype=np.uint32)[0])


def squared_residual_permutation_context(
    e: np.ndarray,
    *,
    n_permutations: int,
    permutation_seed: int,
) -> dict:
    e = np.asarray(e, dtype=float).reshape(-1)
    n = len(e)
    if n < 4 or not np.isfinite(e).all() or n_permutations < 1:
        return {"valid": False, "failure_reason": "invalid permutation context input"}

    rng = np.random.default_rng(permutation_seed)
    permutation_indices = np.vstack(
        [rng.permutation(n) for _ in range(n_permutations)]
    )
    # Cheap sentinels that every generated row contains the full index set.
    expected_sum = n * (n - 1) // 2
    expected_sum_squares = n * (n - 1) * (2 * n - 1) // 6
    assert np.all(permutation_indices.sum(axis=1) == expected_sum)
    assert np.all((permutation_indices**2).sum(axis=1) == expected_sum_squares)

    squared_residual = e**2
    centered = squared_residual - squared_residual.mean()
    total_sum_squares = float(centered @ centered)
    if (
        not np.isfinite(total_sum_squares)
        or total_sum_squares
        <= np.finfo(float).eps * max(1.0, float(squared_residual @ squared_residual))
    ):
        return {"valid": False, "failure_reason": "degenerate squared residuals"}

    permuted = squared_residual[permutation_indices]
    # Recenter explicitly even though permutation preserves the sample mean.
    permuted_centered = permuted - permuted.mean(axis=1, keepdims=True)
    permuted_total_sum_squares = np.einsum(
        "bi,bi->b", permuted_centered, permuted_centered
    )
    sst_max_abs_difference = float(
        np.max(np.abs(permuted_total_sum_squares - total_sum_squares))
    )
    mean_max_abs_difference = float(
        np.max(np.abs(permuted.mean(axis=1) - squared_residual.mean()))
    )
    invariant_tolerance = 1e-10 * max(1.0, total_sum_squares)
    assert sst_max_abs_difference <= invariant_tolerance
    assert mean_max_abs_difference <= 1e-12 * max(1.0, abs(squared_residual.mean()))

    return {
        "valid": True,
        "failure_reason": "",
        "n": n,
        "n_permutations": int(n_permutations),
        "permutation_seed": int(permutation_seed),
        "permutation_indices": permutation_indices,
        "squared_residual": squared_residual,
        "centered_squared_residual": centered,
        "permuted_centered_squared_residual": permuted_centered,
        "total_sum_squares": total_sum_squares,
        "permuted_sst_max_abs_difference": sst_max_abs_difference,
        "permuted_mean_max_abs_difference": mean_max_abs_difference,
    }


def squared_residual_feature_hsic_permutation_test(
    u: np.ndarray,
    e: np.ndarray,
    basis: str,
    permutation_context: dict,
) -> dict:
    u = np.asarray(u, dtype=float).reshape(-1)
    e = np.asarray(e, dtype=float).reshape(-1)
    if not permutation_context.get("valid", False):
        return {
            "valid": False,
            "failure_reason": permutation_context.get(
                "failure_reason", "invalid permutation context"
            ),
        }
    if len(u) != len(e) or len(e) != permutation_context["n"]:
        return {"valid": False, "failure_reason": "incompatible input lengths"}

    prepared = prepare_feature_design(u, basis)
    if not prepared["valid"]:
        return prepared
    n = len(e)
    q = int(prepared["q"])
    if n <= q + 1:
        return {
            "valid": False,
            "failure_reason": "insufficient residual degrees of freedom",
            "q": q,
        }

    # prepare_feature_design stores sqrt(n-1) times the orthonormal left
    # singular vectors, so Q spans the centered, whitened feature space.
    Q = prepared["features"] / np.sqrt(n - 1.0)
    feature_gram = Q.T @ Q
    orthogonality_error = float(
        np.max(np.abs(feature_gram - np.eye(q)))
    )
    # K^2-K = Q (Q'Q-I) Q'. Evaluate its Frobenius norm through q x q
    # factors rather than forming an n x n product for every Monte Carlo row.
    gram_error = feature_gram - np.eye(q)
    idempotence_squared = float(
        np.trace(gram_error.T @ feature_gram @ gram_error @ feature_gram)
    )
    kernel_idempotence_frobenius = math.sqrt(max(0.0, idempotence_squared))
    assert orthogonality_error <= 1e-10
    assert kernel_idempotence_frobenius <= 1e-9

    s = permutation_context["squared_residual"]
    s_centered = permutation_context["centered_squared_residual"]
    total_sum_squares = float(permutation_context["total_sum_squares"])
    projected = Q.T @ s_centered
    statistic = float(projected @ projected)

    # Explicit OLS equivalence check with an intercept and the same feature span.
    coefficients, _, _, _ = np.linalg.lstsq(prepared["design"], s, rcond=None)
    fitted = prepared["design"] @ coefficients
    explained_sum_squares = float(
        np.sum((fitted - s.mean()) ** 2)
    )
    residual_sum_squares = float(np.sum((s - fitted) ** 2))
    r_squared = float(np.clip(statistic / total_sum_squares, 0.0, 1.0))
    ols_r_squared = float(
        np.clip(1.0 - residual_sum_squares / total_sum_squares, 0.0, 1.0)
    )
    equivalence_tolerance = 1e-9 * max(1.0, total_sum_squares)
    ess_absolute_difference = abs(statistic - explained_sum_squares)
    assert ess_absolute_difference <= equivalence_tolerance
    assert abs(r_squared - ols_r_squared) <= 1e-10

    residual_df = n - q - 1
    unexplained = max(0.0, total_sum_squares - statistic)
    if unexplained == 0.0:
        f_statistic = np.inf
    else:
        f_statistic = float((statistic / q) / (unexplained / residual_df))
    if r_squared >= 1.0:
        f_from_r_squared = np.inf
    else:
        f_from_r_squared = float(
            (r_squared / q) / ((1.0 - r_squared) / residual_df)
        )
    if np.isfinite(f_statistic):
        assert abs(f_statistic - f_from_r_squared) <= 1e-9 * max(
            1.0, abs(f_statistic)
        )
    else:
        assert np.isinf(f_from_r_squared)

    permuted_projected = (
        permutation_context["permuted_centered_squared_residual"] @ Q
    )
    null_statistics = np.einsum(
        "bi,bi->b", permuted_projected, permuted_projected
    )
    n_exceedances = int(np.count_nonzero(null_statistics >= statistic))
    n_permutations = int(permutation_context["n_permutations"])
    p_value = float((1 + n_exceedances) / (n_permutations + 1))

    # R^2 and F are fixed monotone transformations of T for this dataset and
    # feature configuration, hence identical permutations must give the same p.
    null_r_squared = null_statistics / total_sum_squares
    r_squared_exceedances = int(np.count_nonzero(null_r_squared >= r_squared))
    r_squared_p_value = float(
        (1 + r_squared_exceedances) / (n_permutations + 1)
    )
    assert r_squared_exceedances == n_exceedances
    assert r_squared_p_value == p_value
    assert np.isclose(
        p_value * (n_permutations + 1),
        1 + n_exceedances,
        rtol=0.0,
        atol=1e-12,
    )

    # The first-order correction for l(e,e')=e^2 e'^2 is exactly zero.
    first_order_correction = 0.0
    debiased_test_statistic = statistic + first_order_correction
    assert debiased_test_statistic == statistic

    valid = (
        np.isfinite(statistic)
        and statistic >= -1e-12
        and np.isfinite(p_value)
        and 0.0 <= p_value <= 1.0
    )
    return {
        "valid": bool(valid),
        "failure_reason": "" if valid else "nonfinite or invalid permutation-HSIC result",
        "statistic": max(0.0, statistic),
        "test_statistic": max(0.0, statistic),
        "p_value": p_value,
        "df": q,
        "q": q,
        "effective_rank": q,
        "raw_n_features": prepared["raw_n_features"],
        "n_permutations": n_permutations,
        "permutation_seed": int(permutation_context["permutation_seed"]),
        "permutation_exceedances": n_exceedances,
        "r_squared": r_squared,
        "ols_r_squared": ols_r_squared,
        "f_statistic": f_statistic,
        "explained_sum_squares": explained_sum_squares,
        "total_sum_squares": total_sum_squares,
        "ess_absolute_difference": ess_absolute_difference,
        "feature_orthogonality_max_abs": orthogonality_error,
        "kernel_idempotence_frobenius": kernel_idempotence_frobenius,
        "permuted_sst_max_abs_difference": permutation_context[
            "permuted_sst_max_abs_difference"
        ],
        "permuted_mean_max_abs_difference": permutation_context[
            "permuted_mean_max_abs_difference"
        ],
        "r_squared_permutation_p_value": r_squared_p_value,
        "first_order_correction": first_order_correction,
        "debiased_test_statistic": debiased_test_statistic,
        "hsic_normalization": HSIC_STATISTIC_NORMALIZATION,
    }


def explicit_projection_kernel_check(
    u: np.ndarray,
    e: np.ndarray,
    basis: str,
    permutation_context: dict,
) -> dict:
    """Materialize K once per basis for visible implementation assertions."""
    prepared = prepare_feature_design(u, basis)
    result = squared_residual_feature_hsic_permutation_test(
        u, e, basis, permutation_context
    )
    assert prepared["valid"] and result["valid"]
    Q = prepared["features"] / np.sqrt(len(u) - 1.0)
    K = Q @ Q.T
    symmetry_error = float(np.max(np.abs(K - K.T)))
    idempotence_error = float(np.max(np.abs(K @ K - K)))
    explicit_statistic = float(
        permutation_context["centered_squared_residual"]
        @ K
        @ permutation_context["centered_squared_residual"]
    )
    effective_rank = int(np.linalg.matrix_rank(Q))
    passed = bool(
        symmetry_error <= 1e-12
        and idempotence_error <= 1e-10
        and effective_rank == prepared["q"]
        and abs(explicit_statistic - result["test_statistic"])
        <= 1e-9 * max(1.0, result["total_sum_squares"])
        and result["ess_absolute_difference"]
        <= 1e-9 * max(1.0, result["total_sum_squares"])
        and result["r_squared_permutation_p_value"] == result["p_value"]
        and result["first_order_correction"] == 0.0
        and result["debiased_test_statistic"] == result["test_statistic"]
    )
    return {
        "basis": basis,
        "effective_rank": effective_rank,
        "kernel_symmetry_max_abs": symmetry_error,
        "kernel_idempotence_max_abs": idempotence_error,
        "explicit_vs_factorized_statistic_abs": abs(
            explicit_statistic - result["test_statistic"]
        ),
        "hsic_vs_ols_ess_abs": result["ess_absolute_difference"],
        "permuted_sst_max_abs_difference": result[
            "permuted_sst_max_abs_difference"
        ],
        "finite_permutation_p_value": result["p_value"],
        "passed": passed,
    }

## 3. Binned Levene and Brown--Forsythe tests

Stable predictor ranks are split with `np.array_split`, so group sizes differ
by at most one and binning never uses residuals.  For bin \(b\), form
\(d_{ib}=|e_{ib}-c_b|\), where \(c_b\) is the bin mean (Levene) or median
(Brown--Forsythe), and apply the ordinary one-way ANOVA \(F\)-test to the
transformed \(d_{ib}\).  A predeclared minimum of five observations per bin
is required; impossible configurations are invalid rather than altered.

In [ ]:
def equal_frequency_groups(z: np.ndarray, n_bins: int) -> tuple[list[np.ndarray], list[int]]:
    z = np.asarray(z, dtype=float).reshape(-1)
    order = np.argsort(z, kind="mergesort")
    index_groups = [np.asarray(group, dtype=int) for group in np.array_split(order, n_bins)]
    counts = [len(group) for group in index_groups]
    flat = np.concatenate(index_groups) if index_groups else np.array([], dtype=int)
    if not np.array_equal(np.sort(flat), np.arange(len(z))):
        raise AssertionError("Rank bins do not form a partition.")
    if counts and max(counts) - min(counts) > 1:
        raise AssertionError("Rank-bin sizes differ by more than one.")
    return index_groups, counts


def binned_dispersion_test(
    z: np.ndarray,
    e: np.ndarray,
    n_bins: int,
    center: str,
) -> dict:
    z = np.asarray(z, dtype=float).reshape(-1)
    e = np.asarray(e, dtype=float).reshape(-1)
    if len(z) != len(e) or not np.isfinite(z).all() or not np.isfinite(e).all():
        return {"valid": False, "failure_reason": "invalid or nonfinite input"}
    groups, counts = equal_frequency_groups(z, n_bins)
    if not counts or min(counts) < MIN_BIN_SIZE:
        return {
            "valid": False,
            "failure_reason": f"minimum bin size {min(counts) if counts else 0} < {MIN_BIN_SIZE}",
            "bin_counts": counts,
        }
    residual_groups = [e[index] for index in groups]
    try:
        statistic, p_value = stats.levene(
            *residual_groups,
            center=center,
            nan_policy="raise",
        )
    except Exception as exc:
        return {
            "valid": False,
            "failure_reason": f"{type(exc).__name__}: {exc}",
            "bin_counts": counts,
        }
    statistic = float(statistic)
    p_value = float(p_value)
    valid = np.isfinite(statistic) and np.isfinite(p_value) and 0.0 <= p_value <= 1.0
    return {
        "valid": bool(valid),
        "failure_reason": "" if valid else "nonfinite or out-of-bounds p-value",
        "statistic": statistic,
        "p_value": p_value,
        "df": n_bins - 1,
        "bin_counts": counts,
    }

## 4. Finite-feature residual quantile regressions

At each requested \(\tau\), fit

\[
Q_\tau(e_i\mid u_i)=a_\tau+\Phi(u_i)^\top b_\tau
\]

with `statsmodels.QuantReg` and its robust covariance estimate.  The joint
slope Wald statistic is computed explicitly from that covariance and checked
against `statsmodels`' own Wald result.  The global p-value is

\[
p_{\rm global}=\min\{1,\ |\mathcal T|\min_{\tau\in\mathcal T}p_\tau\}.
\]

Fits at shared quantiles are computed once and reused between the central and
wide grids.  A grid is invalid if any of its requested quantile fits fails to
converge or has a singular/nonfinite covariance.  First-stage mean error can
affect residual quantiles at first order, so empirical null calibration is
assessed rather than assumed.

In [ ]:
def quantile_wald_test(
    e: np.ndarray,
    prepared: dict,
    tau: float,
) -> dict:
    e = np.asarray(e, dtype=float).reshape(-1)
    if not prepared["valid"]:
        return {
            "valid": False,
            "converged": False,
            "failure_reason": prepared["failure_reason"],
        }
    design = prepared["design"]
    q = prepared["q"]
    if len(e) <= q + 1 or not np.isfinite(e).all():
        return {
            "valid": False,
            "converged": False,
            "failure_reason": "invalid input or insufficient residual degrees of freedom",
        }
    try:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            fit = QuantReg(e, design).fit(
                q=tau,
                vcov="robust",
                kernel="epa",
                bandwidth="hsheather",
                max_iter=QUANTREG_MAX_ITER,
                p_tol=QUANTREG_P_TOL,
            )
        warning_text = " | ".join(str(item.message) for item in caught)
        bad_warning = any(
            issubclass(item.category, (IterationLimitWarning, ConvergenceWarning))
            or "convergence" in str(item.message).lower()
            or "maximum number of iterations" in str(item.message).lower()
            for item in caught
        )
        iterations = int(fit.iterations)
        params = np.asarray(fit.params, dtype=float)
        covariance = np.asarray(fit.cov_params(), dtype=float)
        finite_fit = (
            np.isfinite(params).all()
            and np.isfinite(covariance).all()
            and np.isfinite(float(fit.bandwidth))
            and float(fit.bandwidth) > 0
            and np.isfinite(float(fit.sparsity))
            and float(fit.sparsity) > 0
        )
        converged = (iterations < QUANTREG_MAX_ITER) and not bad_warning
        if not finite_fit:
            raise FloatingPointError("nonfinite parameters, covariance, bandwidth, or sparsity")
        if not converged:
            return {
                "valid": False,
                "converged": False,
                "failure_reason": warning_text or "quantile regression did not converge",
                "iterations": iterations,
                "warning_text": warning_text,
            }
        slopes = params[1:]
        slope_covariance = covariance[1:, 1:]
        slope_covariance = (slope_covariance + slope_covariance.T) / 2.0
        eigenvalues = np.linalg.eigvalsh(slope_covariance)
        covariance_tolerance = (
            max(1.0, float(np.max(np.abs(eigenvalues))))
            * q
            * np.finfo(float).eps
            * 100
        )
        if np.min(eigenvalues) <= covariance_tolerance:
            return {
                "valid": False,
                "converged": True,
                "failure_reason": "robust slope covariance is singular or not positive definite",
                "iterations": iterations,
                "warning_text": warning_text,
            }
        statistic = float(slopes @ linalg.solve(slope_covariance, slopes, assume_a="pos"))
        p_value = float(chi2.sf(statistic, q))
        restriction = np.column_stack([np.zeros(q), np.eye(q)])
        reference = fit.wald_test(restriction, use_f=False, scalar=True)
        reference_p_value = float(np.asarray(reference.pvalue))
        wald_agreement = abs(p_value - reference_p_value)
        valid = (
            np.isfinite(statistic)
            and statistic >= -1e-12
            and np.isfinite(p_value)
            and 0.0 <= p_value <= 1.0
            and wald_agreement <= 1e-8
        )
        return {
            "valid": bool(valid),
            "converged": True,
            "failure_reason": "" if valid else "invalid Wald result or statsmodels disagreement",
            "statistic": max(0.0, statistic),
            "p_value": p_value,
            "df": q,
            "iterations": iterations,
            "warning_text": warning_text,
            "bandwidth": float(fit.bandwidth),
            "sparsity": float(fit.sparsity),
            "statsmodels_p_value": reference_p_value,
            "wald_agreement": wald_agreement,
        }
    except Exception as exc:
        return {
            "valid": False,
            "converged": False,
            "failure_reason": f"{type(exc).__name__}: {exc}",
            "warning_text": locals().get("warning_text", ""),
        }


def quantile_grid_tests(u: np.ndarray, e: np.ndarray, basis: str) -> dict[str, dict]:
    prepared = prepare_feature_design(u, basis)
    per_tau = {
        tau: quantile_wald_test(e, prepared, tau)
        for tau in ALL_QUANTILES
    }
    output = {}
    for grid_name, grid in QUANTILE_GRIDS.items():
        selected = {tau: per_tau[tau] for tau in grid}
        valid = prepared["valid"] and all(item["valid"] for item in selected.values())
        if valid:
            p_values = {tau: float(item["p_value"]) for tau, item in selected.items()}
            global_p = float(min(1.0, len(grid) * min(p_values.values())))
            failure_reason = ""
        else:
            p_values = {
                tau: (float(item["p_value"]) if item.get("valid") else None)
                for tau, item in selected.items()
            }
            global_p = np.nan
            failure_reason = " | ".join(
                f"tau={tau:g}: {item['failure_reason']}"
                for tau, item in selected.items()
                if not item["valid"]
            )
        output[grid_name] = {
            "valid": bool(valid),
            "failure_reason": failure_reason,
            "p_value": global_p,
            "q": prepared.get("q", 0),
            "df": prepared.get("q", 0),
            "component_p_values": p_values,
            "component_iterations": {
                tau: item.get("iterations") for tau, item in selected.items()
            },
            "component_converged": {
                tau: bool(item.get("converged", False)) for tau, item in selected.items()
            },
            "component_warnings": {
                tau: item.get("warning_text", "") for tau, item in selected.items()
            },
            "max_wald_agreement": max(
                (
                    item.get("wald_agreement", 0.0)
                    for item in selected.values()
                    if item.get("valid")
                ),
                default=np.nan,
            ),
        }
    return output

## Common residual construction

`fit_mean_forecasts` and `fit_mean_on_split` below are the exact functions
used by the HSIC baseline.  A single standardized full predictor is carried
into each residual scheme.  Explicit index audits verify that split
evaluation observations are absent from training and that every cross-fit
prediction comes from a model that excluded that observation.

In [ ]:
def construct_residual_sets(data: dict, repo_direction: str) -> tuple[list[dict], list[dict]]:
    predictor, response, signal = direction_arrays(data, repo_direction)
    predictor = np.asarray(predictor, dtype=float)
    response = np.asarray(response, dtype=float)
    signal = np.asarray(signal, dtype=float)
    n = len(response)
    all_indices = np.arange(n)
    predictor_sd = predictor.std(ddof=1)
    if not np.isfinite(predictor_sd) or predictor_sd <= 0:
        raise RuntimeError("Predictor standard deviation is invalid.")
    u_full = (predictor - predictor.mean()) / predictor_sd

    X = jnp.asarray(predictor[:, None])
    Y = jnp.asarray(response[:, None])
    signal_array = jnp.asarray(signal[:, None])
    fold_masks = np.asarray(build_fold_masks(n, N_FOLDS), dtype=bool)
    residual_sets = []
    audit_rows = []

    forecasts, mean_diagnostics = fit_mean_forecasts(
        CORE_CONFIG,
        X,
        Y,
        signal_array,
        jnp.asarray(fold_masks),
    )
    forecasts = np.asarray(forecasts)[:, :, 0]
    crossfit_residual = np.empty(n, dtype=float)
    coverage = np.zeros(n, dtype=int)
    for fold_index, holdout_mask in enumerate(fold_masks):
        holdout = all_indices[holdout_mask]
        train = all_indices[~holdout_mask]
        if np.intersect1d(train, holdout).size:
            raise AssertionError("Cross-fit training and holdout indices overlap.")
        crossfit_residual[holdout] = response[holdout] - forecasts[fold_index, holdout]
        coverage[holdout] += 1
        audit_rows.append(
            {
                "check": "crossfit exclusion",
                "scheme": "cross_fitted_variant",
                "fold_or_prefix": fold_index,
                "n_train": len(train),
                "n_eval": len(holdout),
                "overlap": int(np.intersect1d(train, holdout).size),
                "passed": True,
            }
        )
    if not np.array_equal(coverage, np.ones(n, dtype=int)):
        raise AssertionError("Cross-fit holdouts do not cover every observation exactly once.")
    residual_sets.append(
        {
            "residual_scheme": "cross_fitted_variant",
            "scheme_label": "cross-fit (5 folds)",
            "training_fraction": (N_FOLDS - 1) / N_FOLDS,
            "n_folds": N_FOLDS,
            "eval_indices": all_indices,
            "predictor": predictor,
            "u": u_full,
            "residual": crossfit_residual,
            "mean_diagnostics": mean_diagnostics,
        }
    )

    for prefix in SPLIT_PREFIXES:
        train_mask = np.any(fold_masks[:prefix], axis=0)
        eval_mask = ~train_mask
        train = all_indices[train_mask]
        evaluation = all_indices[eval_mask]
        if np.intersect1d(train, evaluation).size:
            raise AssertionError("Sample-split training and evaluation indices overlap.")
        _, pred_eval, mean_diagnostic = fit_mean_on_split(
            CORE_CONFIG,
            X,
            Y,
            signal_array,
            train_mask,
            eval_mask,
            split_id=prefix,
        )
        residual = response[evaluation] - np.asarray(pred_eval)[:, 0]
        fraction = prefix / N_FOLDS
        residual_sets.append(
            {
                "residual_scheme": "sample_split",
                "scheme_label": f"split {int(100 * fraction)}% train",
                "training_fraction": fraction,
                "n_folds": np.nan,
                "eval_indices": evaluation,
                "predictor": predictor[evaluation],
                "u": u_full[evaluation],
                "residual": residual,
                "mean_diagnostics": [mean_diagnostic],
            }
        )
        audit_rows.append(
            {
                "check": "sample-split exclusion",
                "scheme": "sample_split",
                "fold_or_prefix": prefix,
                "n_train": len(train),
                "n_eval": len(evaluation),
                "overlap": int(np.intersect1d(train, evaluation).size),
                "passed": True,
            }
        )
    return residual_sets, audit_rows


def scenario_label(rho: float, direction: str) -> str:
    rho_text = "0" if np.isclose(rho, 0.0) else "0.35"
    role = "size" if np.isclose(rho, 0.0) and direction == "X->Y" else "power"
    return f"rho={rho_text}, {direction} ({role})"


SCENARIO_ORDER = [
    "rho=0, X->Y (size)",
    "rho=0, Y->X (power)",
    "rho=0.35, X->Y (power)",
    "rho=0.35, Y->X (power)",
]


def base_result_record(
    *,
    rho: float,
    repo_direction: str,
    repetition: int,
    data_seed: int,
    residual_set: dict,
) -> dict:
    direction = DIRECTION_MAP[repo_direction]
    return {
        "rho": float(rho),
        "direction": direction,
        "null_truth": bool(np.isclose(rho, 0.0) and direction == "X->Y"),
        "method": "",
        "residual_scheme": residual_set["residual_scheme"],
        "scheme_label": residual_set["scheme_label"],
        "training_fraction": float(residual_set["training_fraction"]),
        "n_folds": residual_set["n_folds"],
        "feature_basis": "",
        "n_features": np.nan,
        "effective_rank": np.nan,
        "n_bins": np.nan,
        "quantile_grid": "",
        "repetition": int(repetition),
        "data_seed": int(data_seed),
        "n_eval": int(len(residual_set["residual"])),
        "p_value": np.nan,
        "reject": np.nan,
        "valid": False,
        "failure_reason": "",
        "statistic": np.nan,
        "test_statistic": np.nan,
        "df": np.nan,
        "n_permutations": np.nan,
        "permutation_seed": np.nan,
        "permutation_exceedances": np.nan,
        "r_squared": np.nan,
        "ols_r_squared": np.nan,
        "f_statistic": np.nan,
        "explained_sum_squares": np.nan,
        "total_sum_squares": np.nan,
        "ess_absolute_difference": np.nan,
        "feature_orthogonality_max_abs": np.nan,
        "kernel_idempotence_frobenius": np.nan,
        "permuted_sst_max_abs_difference": np.nan,
        "permuted_mean_max_abs_difference": np.nan,
        "r_squared_permutation_p_value": np.nan,
        "first_order_correction": np.nan,
        "debiased_test_statistic": np.nan,
        "hsic_normalization": "",
        "bin_counts": "",
        "quantile_p_values": "",
        "quantile_iterations": "",
        "quantile_converged": "",
        "quantile_warnings": "",
        "max_wald_agreement": np.nan,
    }


def finish_record(record: dict, result: dict) -> dict:
    valid = bool(result.get("valid", False))
    p_value = float(result["p_value"]) if valid else np.nan
    if valid and (not np.isfinite(p_value) or not 0.0 <= p_value <= 1.0):
        valid = False
        p_value = np.nan
        result = {**result, "failure_reason": "final p-value failed bounds check"}
    record.update(
        {
            "p_value": p_value,
            "reject": bool(p_value < ALPHA) if valid else np.nan,
            "valid": valid,
            "failure_reason": result.get("failure_reason", ""),
            "statistic": result.get("statistic", np.nan),
            "test_statistic": result.get(
                "test_statistic", result.get("statistic", np.nan)
            ),
            "df": result.get("df", np.nan),
        }
    )
    return record


def diagnostics_for_residual(
    *,
    rho: float,
    repo_direction: str,
    repetition: int,
    data_seed: int,
    residual_set: dict,
) -> list[dict]:
    u = residual_set["u"]
    z = residual_set["predictor"]
    e = residual_set["residual"]
    records = []

    permutation_seed = permutation_seed_for(
        data_seed, repo_direction, residual_set
    )
    permutation_context = squared_residual_permutation_context(
        e,
        n_permutations=N_PERMUTATIONS,
        permutation_seed=permutation_seed,
    )

    # Interleave analytic and permutation calibration for the identical basis,
    # predictor, and residual vector. The shared context contains the one
    # permutation-index array reused by all seven bases.
    for basis in VARIANCE_BASES:
        result = variance_score_test(u, e, basis)
        record = base_result_record(
            rho=rho,
            repo_direction=repo_direction,
            repetition=repetition,
            data_seed=data_seed,
            residual_set=residual_set,
        )
        record.update(
            {
                "method": "variance_score",
                "feature_basis": basis,
                "n_features": result.get("q", np.nan),
                "effective_rank": result.get("q", np.nan),
            }
        )
        records.append(finish_record(record, result))

        permutation_result = squared_residual_feature_hsic_permutation_test(
            u, e, basis, permutation_context
        )
        permutation_record = base_result_record(
            rho=rho,
            repo_direction=repo_direction,
            repetition=repetition,
            data_seed=data_seed,
            residual_set=residual_set,
        )
        permutation_record.update(
            {
                "method": "squared_residual_feature_hsic_permutation",
                "feature_basis": basis,
                "n_features": permutation_result.get("q", np.nan),
                "effective_rank": permutation_result.get(
                    "effective_rank", np.nan
                ),
                "n_permutations": permutation_result.get(
                    "n_permutations", N_PERMUTATIONS
                ),
                "permutation_seed": permutation_result.get(
                    "permutation_seed", permutation_seed
                ),
                "permutation_exceedances": permutation_result.get(
                    "permutation_exceedances", np.nan
                ),
                "r_squared": permutation_result.get("r_squared", np.nan),
                "ols_r_squared": permutation_result.get(
                    "ols_r_squared", np.nan
                ),
                "f_statistic": permutation_result.get(
                    "f_statistic", np.nan
                ),
                "explained_sum_squares": permutation_result.get(
                    "explained_sum_squares", np.nan
                ),
                "total_sum_squares": permutation_result.get(
                    "total_sum_squares", np.nan
                ),
                "ess_absolute_difference": permutation_result.get(
                    "ess_absolute_difference", np.nan
                ),
                "feature_orthogonality_max_abs": permutation_result.get(
                    "feature_orthogonality_max_abs", np.nan
                ),
                "kernel_idempotence_frobenius": permutation_result.get(
                    "kernel_idempotence_frobenius", np.nan
                ),
                "permuted_sst_max_abs_difference": permutation_result.get(
                    "permuted_sst_max_abs_difference", np.nan
                ),
                "permuted_mean_max_abs_difference": permutation_result.get(
                    "permuted_mean_max_abs_difference", np.nan
                ),
                "r_squared_permutation_p_value": permutation_result.get(
                    "r_squared_permutation_p_value", np.nan
                ),
                "first_order_correction": permutation_result.get(
                    "first_order_correction", np.nan
                ),
                "debiased_test_statistic": permutation_result.get(
                    "debiased_test_statistic", np.nan
                ),
                "hsic_normalization": permutation_result.get(
                    "hsic_normalization", ""
                ),
            }
        )
        records.append(finish_record(permutation_record, permutation_result))

    for n_bins in BIN_COUNTS:
        for method, center in (("levene", "mean"), ("brown_forsythe", "median")):
            result = binned_dispersion_test(z, e, n_bins, center)
            record = base_result_record(
                rho=rho,
                repo_direction=repo_direction,
                repetition=repetition,
                data_seed=data_seed,
                residual_set=residual_set,
            )
            record.update(
                {
                    "method": method,
                    "n_bins": n_bins,
                    "bin_counts": json.dumps(result.get("bin_counts", [])),
                }
            )
            records.append(finish_record(record, result))

    for basis in QUANTILE_BASES:
        grid_results = quantile_grid_tests(u, e, basis)
        for grid_name, result in grid_results.items():
            record = base_result_record(
                rho=rho,
                repo_direction=repo_direction,
                repetition=repetition,
                data_seed=data_seed,
                residual_set=residual_set,
            )
            record.update(
                {
                    "method": "quantile_regression",
                    "feature_basis": basis,
                    "n_features": result.get("q", np.nan),
                    "quantile_grid": grid_name,
                    "quantile_p_values": json.dumps(result["component_p_values"]),
                    "quantile_iterations": json.dumps(result["component_iterations"]),
                    "quantile_converged": json.dumps(result["component_converged"]),
                    "quantile_warnings": json.dumps(result["component_warnings"]),
                    "max_wald_agreement": result.get("max_wald_agreement", np.nan),
                }
            )
            records.append(finish_record(record, result))
    return records

## Direct-error implementation checks

These checks use *observed* Gaussian errors independent of the predictor, so
there is no first-stage estimation.  Every p-value configuration is run at a
comfortably asymptotic \(n=1000\) over a modest Monte Carlo.  Exact 5% in
each row is not expected: the two quantile grids use conservative Bonferroni
combination, while some asymptotic robust-Wald and finite-group \(F\)
references can still be imperfect.  The hard checks are that all
configurations are valid and p-values are finite and bounded.  A binomial
near-nominal flag is reported rather than imposed on every configuration:
the required high-dimensional robust-Wald reference can itself be liberal,
and discovering that failure is part of the exercise.  As implementation
sentinels, the variance-score and binned families, plus the low-dimensional
linear quantile tests, are required not to reject above a predeclared 0.12
threshold.  This smoke exercise is not used to calibrate the tests.

A separate strong sinusoidal scale alternative must give high power to the
predeclared matching `fourier2` variance-score basis.

In [ ]:
SMOKE_REPS = int(os.environ.get("CLASSICAL_DIAGNOSTICS_SMOKE_REPS", "200"))
SMOKE_N = 1_000
STRONG_ALT_REPS = int(os.environ.get("CLASSICAL_DIAGNOSTICS_POWER_REPS", "100"))
SMOKE_SEED = 20260724


def direct_error_smoke(n_reps: int = SMOKE_REPS, n: int = SMOKE_N) -> pd.DataFrame:
    rng = np.random.default_rng(SMOKE_SEED)
    rows = []
    for repetition in range(n_reps):
        z = rng.normal(size=n)
        e = rng.normal(size=n)
        u = (z - z.mean()) / z.std(ddof=1)
        for basis in VARIANCE_BASES:
            result = variance_score_test(u, e, basis)
            rows.append(
                {
                    "method": "variance_score",
                    "configuration": basis,
                    "repetition": repetition,
                    **result,
                }
            )
        for n_bins in BIN_COUNTS:
            for method, center in (("levene", "mean"), ("brown_forsythe", "median")):
                result = binned_dispersion_test(z, e, n_bins, center)
                rows.append(
                    {
                        "method": method,
                        "configuration": f"B={n_bins}",
                        "repetition": repetition,
                        **result,
                    }
                )
        for basis in QUANTILE_BASES:
            for grid_name, result in quantile_grid_tests(u, e, basis).items():
                rows.append(
                    {
                        "method": "quantile_regression",
                        "configuration": f"{basis}, {grid_name}",
                        "repetition": repetition,
                        **result,
                    }
                )
    frame = pd.DataFrame(rows)
    frame["reject"] = np.where(frame["valid"], frame["p_value"] < ALPHA, np.nan)
    summary = (
        frame.groupby(["method", "configuration"], sort=False, dropna=False)
        .agg(
            repetitions=("repetition", "nunique"),
            valid=("valid", "sum"),
            rejection_rate=("reject", "mean"),
            min_p_value=("p_value", "min"),
            max_p_value=("p_value", "max"),
        )
        .reset_index()
    )
    summary["mcse"] = np.sqrt(
        summary["rejection_rate"]
        * (1.0 - summary["rejection_rate"])
        / summary["valid"]
    )
    nominal_mcse = np.sqrt(ALPHA * (1.0 - ALPHA) / summary["valid"])
    summary["near_nominal_95_band"] = (
        (summary["rejection_rate"] >= ALPHA - 1.96 * nominal_mcse)
        & (summary["rejection_rate"] <= ALPHA + 1.96 * nominal_mcse)
    )
    summary["material_excess_flag"] = summary["rejection_rate"] > 0.10
    return summary


smoke_started = time.perf_counter()
null_smoke = direct_error_smoke()
null_smoke.to_csv(NULL_SMOKE_CSV, index=False)
display(null_smoke)

assert (null_smoke["valid"] == null_smoke["repetitions"]).all()
assert null_smoke[["min_p_value", "max_p_value"]].notna().all().all()
assert (null_smoke["min_p_value"] >= 0).all() and (null_smoke["max_p_value"] <= 1).all()
assert null_smoke.loc[
    null_smoke["method"].isin(["variance_score", "levene", "brown_forsythe"]),
    "rejection_rate",
].max() <= 0.12
assert null_smoke.loc[
    (null_smoke["method"] == "quantile_regression")
    & null_smoke["configuration"].isin(["linear, central", "linear, wide"]),
    "rejection_rate",
].max() <= 0.12
if null_smoke["material_excess_flag"].any():
    flagged = null_smoke.loc[
        null_smoke["material_excess_flag"],
        ["method", "configuration", "rejection_rate", "mcse"],
    ]
    print(
        "Direct-error configurations with material asymptotic-reference "
        "size distortion (reported, not recalibrated):"
    )
    display(flagged)


def strong_variance_smoke(n_reps: int = STRONG_ALT_REPS, n: int = 500) -> pd.DataFrame:
    rng = np.random.default_rng(SMOKE_SEED + 1)
    rows = []
    for repetition in range(n_reps):
        z = rng.uniform(-2.0, 2.0, size=n)
        u = (z - z.mean()) / z.std(ddof=1)
        # Strong, positive sinusoidal scale; no DGP coefficient or phase is used.
        e = np.exp(0.9 * np.sin(2.0 * u)) * rng.normal(size=n)
        result = variance_score_test(u, e, "fourier2")
        rows.append(
            {
                "repetition": repetition,
                "valid": result["valid"],
                "p_value": result.get("p_value", np.nan),
                "reject": result.get("p_value", np.nan) < ALPHA
                if result["valid"]
                else np.nan,
                "failure_reason": result.get("failure_reason", ""),
            }
        )
    return pd.DataFrame(rows)


strong_smoke = strong_variance_smoke()
strong_smoke.to_csv(POWER_SMOKE_CSV, index=False)
strong_power = float(strong_smoke.loc[strong_smoke["valid"], "reject"].mean())
strong_mcse = math.sqrt(strong_power * (1 - strong_power) / strong_smoke["valid"].sum())
print(
    f"Strong sinusoidal variance alternative, fourier2: "
    f"power={strong_power:.3f} (MCSE {strong_mcse:.3f}), "
    f"valid={int(strong_smoke['valid'].sum())}/{len(strong_smoke)}"
)
assert strong_smoke["valid"].all()
assert strong_power >= 0.80
print(f"Direct-error checks finished in {time.perf_counter() - smoke_started:.1f}s.")

## Oracle permutation-calibration sanity check

The DGP exposes the true forward structural noise as `data["eps"]`. In this
separate experiment, only the true null `rho=0, X->Y` is considered and
\(s=\varepsilon^2\) is used directly. Here the predictor and squared noise
are independent, so permutation calibration is finite-sample exact
conditional on the observed values (up to the finite 399-permutation Monte
Carlo grid). These oracle rows are saved separately and never substituted for
feasible estimated-residual results.

In [ ]:
ORACLE_PERMUTATION_REPS = int(
    os.environ.get("CLASSICAL_DIAGNOSTICS_ORACLE_REPS", "200")
)
ORACLE_PERMUTATION_SEED = 20_260_725


def run_oracle_permutation_sanity(n_reps: int) -> pd.DataFrame:
    rows = []
    for repetition in range(n_reps):
        data = simulate_anm(
            CORE_N,
            rho=0.0,
            seed=ORACLE_PERMUTATION_SEED + repetition,
        )
        z = np.asarray(data["x"], dtype=float)
        epsilon = np.asarray(data["eps"], dtype=float)
        u = (z - z.mean()) / z.std(ddof=1)
        permutation_seed = int(
            np.random.SeedSequence(
                [PERMUTATION_SEED_BASE, ORACLE_PERMUTATION_SEED, repetition]
            ).generate_state(1, dtype=np.uint32)[0]
        )
        context = squared_residual_permutation_context(
            epsilon,
            n_permutations=N_PERMUTATIONS,
            permutation_seed=permutation_seed,
        )
        for basis in HSIC_FEATURE_BASES:
            result = squared_residual_feature_hsic_permutation_test(
                u, epsilon, basis, context
            )
            rows.append(
                {
                    "method": "squared_residual_feature_hsic_permutation_oracle",
                    "feature_basis": basis,
                    "effective_rank": result.get("effective_rank", np.nan),
                    "residual_scheme": "oracle_structural_noise",
                    "training_fraction": np.nan,
                    "n_folds": np.nan,
                    "rho": 0.0,
                    "direction": "X->Y",
                    "repetition": repetition,
                    "data_seed": ORACLE_PERMUTATION_SEED + repetition,
                    "test_statistic": result.get("test_statistic", np.nan),
                    "p_value": result.get("p_value", np.nan),
                    "reject": (
                        bool(result["p_value"] < ALPHA)
                        if result.get("valid", False)
                        else np.nan
                    ),
                    "n_permutations": result.get("n_permutations", np.nan),
                    "permutation_seed": permutation_seed,
                    "valid": bool(result.get("valid", False)),
                    "failure_reason": result.get("failure_reason", ""),
                }
            )
    return pd.DataFrame(rows)


oracle_permutation_long = run_oracle_permutation_sanity(
    ORACLE_PERMUTATION_REPS
)
assert len(oracle_permutation_long) == (
    ORACLE_PERMUTATION_REPS * len(HSIC_FEATURE_BASES)
)
assert oracle_permutation_long["valid"].all()
assert oracle_permutation_long["p_value"].between(0, 1).all()
assert oracle_permutation_long["n_permutations"].eq(N_PERMUTATIONS).all()
assert np.allclose(
    oracle_permutation_long["p_value"] * (N_PERMUTATIONS + 1),
    np.round(
        oracle_permutation_long["p_value"] * (N_PERMUTATIONS + 1)
    ),
)

oracle_summary_rows = []
for basis, frame in oracle_permutation_long.groupby(
    "feature_basis", sort=False
):
    p_values = frame["p_value"].to_numpy(dtype=float)
    n_valid = len(p_values)
    rejections = int(np.sum(p_values < ALPHA))
    rejection_rate = rejections / n_valid
    mcse = math.sqrt(rejection_rate * (1.0 - rejection_rate) / n_valid)
    interval = stats.binomtest(rejections, n_valid).proportion_ci(
        confidence_level=0.95,
        method="exact",
    )
    uniform_check = stats.kstest(p_values, "uniform")
    oracle_summary_rows.append(
        {
            "feature_basis": basis,
            "effective_rank": int(frame["effective_rank"].iloc[0]),
            "n_valid": n_valid,
            "rejections": rejections,
            "rejection_rate": rejection_rate,
            "mcse": mcse,
            "ci_lower": float(interval.low),
            "ci_upper": float(interval.high),
            "mean_p_value": float(np.mean(p_values)),
            "uniform_ks_statistic": float(uniform_check.statistic),
            "uniform_ks_p_value": float(uniform_check.pvalue),
        }
    )
oracle_permutation_summary = pd.DataFrame(oracle_summary_rows)

# Fixed-seed, deliberately loose implementation sentinels: every exact-null
# row must be near nominal and show no gross departure from uniformity.
assert oracle_permutation_summary["rejection_rate"].between(0.0, 0.11).all()
assert oracle_permutation_summary["mean_p_value"].between(0.40, 0.60).all()
assert oracle_permutation_summary["uniform_ks_p_value"].ge(1e-3).all()

oracle_permutation_long.to_csv(ORACLE_PERMUTATION_LONG_CSV, index=False)
oracle_permutation_summary.to_csv(
    ORACLE_PERMUTATION_SUMMARY_CSV, index=False
)
display(oracle_permutation_summary)
print(
    f"Oracle permutation sanity check: {len(oracle_permutation_long):,} rows; "
    f"rejection range "
    f"{oracle_permutation_summary['rejection_rate'].min():.3f}--"
    f"{oracle_permutation_summary['rejection_rate'].max():.3f}."
)

## Residual-exclusion and exact HSIC-baseline checks

Before the full run, the next cell verifies:

- split residuals contain only evaluation observations;
- every cross-fit observation is predicted by a model excluding it;
- the constructed cross-fit residuals equal those saved by the current HSIC
  baseline for the repository's demonstration seed.

In [ ]:
verification_rows = []
demo_data = simulate_anm(CORE_N, rho=0.0, seed=SEED)
demo_residual_sets, demo_audits = construct_residual_sets(demo_data, "Y on X")
verification_rows.extend(demo_audits)

crossfit_demo = next(
    item for item in demo_residual_sets if item["residual_scheme"] == "cross_fitted_variant"
)
with BASELINE_RUN_JSON.open("r", encoding="utf-8") as handle:
    baseline_payload = json.load(handle)
baseline_residual = np.asarray(
    baseline_payload["diagnostics"]["(0.0, 'Y on X')"]["residual_oos"],
    dtype=float,
)
max_residual_difference = float(
    np.max(np.abs(crossfit_demo["residual"] - baseline_residual))
)
exact_residual_match = bool(np.array_equal(crossfit_demo["residual"], baseline_residual))
verification_rows.append(
    {
        "check": "saved HSIC residual equality",
        "scheme": "cross_fitted_variant",
        "fold_or_prefix": "all",
        "n_train": 200,
        "n_eval": CORE_N,
        "overlap": 0,
        "passed": exact_residual_match,
        "max_absolute_difference": max_residual_difference,
    }
)

# Exercise all p-value functions once on feasible estimated residuals.
one_residual_records = diagnostics_for_residual(
    rho=0.0,
    repo_direction="Y on X",
    repetition=-1,
    data_seed=SEED,
    residual_set=crossfit_demo,
)
one_residual_frame = pd.DataFrame(one_residual_records)

# Materialize the projection kernel for every declared basis on one feasible
# residual vector. Full Monte Carlo rows use the equivalent thin-Q factorization.
demo_permutation_seed = permutation_seed_for(
    SEED, "Y on X", crossfit_demo
)
demo_permutation_context = squared_residual_permutation_context(
    crossfit_demo["residual"],
    n_permutations=N_PERMUTATIONS,
    permutation_seed=demo_permutation_seed,
)
projection_check_rows = [
    explicit_projection_kernel_check(
        crossfit_demo["u"],
        crossfit_demo["residual"],
        basis,
        demo_permutation_context,
    )
    for basis in HSIC_FEATURE_BASES
]
for item in projection_check_rows:
    verification_rows.append(
        {
            "check": f"projection-HSIC equivalence: {item['basis']}",
            "scheme": "cross_fitted_variant",
            "fold_or_prefix": "all",
            "n_train": 200,
            "n_eval": CORE_N,
            "overlap": 0,
            **item,
        }
    )

permutation_demo = one_residual_frame.loc[
    one_residual_frame["method"]
    == "squared_residual_feature_hsic_permutation"
]
assert len(permutation_demo) == len(HSIC_FEATURE_BASES)
assert permutation_demo["valid"].all()
assert permutation_demo["permutation_seed"].nunique() == 1
assert permutation_demo["n_permutations"].eq(N_PERMUTATIONS).all()
assert np.allclose(
    permutation_demo["p_value"] * (N_PERMUTATIONS + 1),
    np.round(permutation_demo["p_value"] * (N_PERMUTATIONS + 1)),
)
assert np.allclose(
    permutation_demo["p_value"],
    permutation_demo["r_squared_permutation_p_value"],
)
assert permutation_demo["first_order_correction"].eq(0.0).all()
assert np.allclose(
    permutation_demo["test_statistic"],
    permutation_demo["debiased_test_statistic"],
)
assert all(item["passed"] for item in projection_check_rows)

verification_rows.append(
    {
        "check": "one-dataset ranks and p-value bounds",
        "scheme": "all diagnostics",
        "fold_or_prefix": "all",
        "n_train": 200,
        "n_eval": CORE_N,
        "overlap": 0,
        "passed": bool(
            one_residual_frame["valid"].all()
            and one_residual_frame["p_value"].between(0, 1).all()
            and one_residual_frame.loc[
                one_residual_frame["feature_basis"] != "", "n_features"
            ].ge(1).all()
        ),
    }
)
max_wald_difference = float(
    one_residual_frame["max_wald_agreement"].dropna().max()
)
verification_rows.append(
    {
        "check": "manual vs statsmodels robust Wald",
        "scheme": "quantile_regression",
        "fold_or_prefix": "all",
        "n_train": 200,
        "n_eval": CORE_N,
        "overlap": 0,
        "passed": max_wald_difference <= 1e-8,
        "max_absolute_difference": max_wald_difference,
    }
)

verification = pd.DataFrame(verification_rows)
verification.to_csv(VERIFICATION_CSV, index=False)
display(verification)
assert verification["passed"].all()
assert exact_residual_match and max_residual_difference == 0.0

## Paired 100-repetition experiment

The faithful count is a visible configuration variable below.  An
environment override exists only for developer smoke runs and emits a
conspicuous warning; the executed deliverable must and does use all 100
repetitions.  Within each `(repetition, rho, direction)`, first-stage
residuals are constructed once, then reused by every second-stage
configuration.

In [ ]:
FULL_N_REPS = int(
    os.environ.get("CLASSICAL_DIAGNOSTICS_N_REPS", str(CORE_N_REPS))
)
if FULL_N_REPS != CORE_N_REPS:
    warnings.warn(
        f"NON-FAITHFUL DEVELOPER OVERRIDE: running {FULL_N_REPS} rather than "
        f"the authoritative {CORE_N_REPS} repetitions.",
        RuntimeWarning,
    )
else:
    print(f"Faithful full run: {FULL_N_REPS} repetitions.")


def run_full_experiment(n_reps: int) -> tuple[pd.DataFrame, list[dict], float]:
    records = []
    all_audits = []
    started = time.perf_counter()
    for repetition in range(n_reps):
        repetition_started = time.perf_counter()
        for rho_index, rho in enumerate(RHO_VALUES):
            data_seed = DATA_SEED_BASE + 1_000 * repetition + 100 * rho_index
            # Exactly one paired dataset per (repetition, rho).
            data = simulate_anm(CORE_N, rho, seed=data_seed)
            for repo_direction in REPO_DIRECTIONS:
                residual_sets, audits = construct_residual_sets(data, repo_direction)
                all_audits.extend(
                    {
                        **audit,
                        "repetition": repetition,
                        "rho": rho,
                        "direction": DIRECTION_MAP[repo_direction],
                    }
                    for audit in audits
                )
                for residual_set in residual_sets:
                    records.extend(
                        diagnostics_for_residual(
                            rho=rho,
                            repo_direction=repo_direction,
                            repetition=repetition,
                            data_seed=data_seed,
                            residual_set=residual_set,
                        )
                    )
        elapsed = time.perf_counter() - started
        progress = {
            "status": "running",
            "completed_repetitions": repetition + 1,
            "target_repetitions": n_reps,
            "rows": len(records),
            "elapsed_seconds": elapsed,
            "last_repetition_seconds": time.perf_counter() - repetition_started,
            "updated_at_utc": datetime.now(timezone.utc).isoformat(),
        }
        with PROGRESS_JSON.open("w", encoding="utf-8") as handle:
            json.dump(progress, handle, indent=2)
        print(
            f"completed repetition {repetition + 1:3d}/{n_reps} | "
            f"rows={len(records):6d} | elapsed={elapsed / 60:.1f} min",
            flush=True,
        )
    runtime = time.perf_counter() - started
    return pd.DataFrame(records), all_audits, runtime


long_results, full_audits, full_runtime_seconds = run_full_experiment(FULL_N_REPS)
with PROGRESS_JSON.open("w", encoding="utf-8") as handle:
    json.dump(
        {
            "status": "complete",
            "completed_repetitions": FULL_N_REPS,
            "target_repetitions": FULL_N_REPS,
            "rows": len(long_results),
            "elapsed_seconds": full_runtime_seconds,
            "updated_at_utc": datetime.now(timezone.utc).isoformat(),
        },
        handle,
        indent=2,
    )

print(
    f"Full experiment complete: {len(long_results):,} rows in "
    f"{full_runtime_seconds / 60:.1f} minutes."
)

## Full-run validation and long-form results

The expected row count is

\[
100\times4\ {\rm scenarios}\times5\ {\rm residual\ schemes}
\times(7+7+6+6+10)=72{,}000,
\]

where the second seven are the new permutation-HSIC counterparts to the
seven analytic variance scores. Invalid fits are retained with an explicit
reason and a missing rejection indicator. No failed configuration is
silently counted as a non-rejection.

In [ ]:
REQUIRED_COLUMNS = [
    "rho",
    "direction",
    "null_truth",
    "method",
    "residual_scheme",
    "training_fraction",
    "n_folds",
    "feature_basis",
    "n_features",
    "effective_rank",
    "n_bins",
    "quantile_grid",
    "repetition",
    "test_statistic",
    "p_value",
    "reject",
    "n_permutations",
    "valid",
    "failure_reason",
]
expected_rows = FULL_N_REPS * 4 * 5 * (
    2 * len(VARIANCE_BASES)
    + 2 * len(BIN_COUNTS)
    + len(QUANTILE_BASES) * len(QUANTILE_GRIDS)
)
assert len(long_results) == expected_rows
assert set(REQUIRED_COLUMNS).issubset(long_results.columns)
assert long_results["repetition"].nunique() == FULL_N_REPS
assert set(long_results["rho"]) == set(RHO_VALUES)
assert set(long_results["direction"]) == {"X->Y", "Y->X"}
assert not long_results.loc[long_results["valid"], "p_value"].isna().any()
assert long_results.loc[long_results["valid"], "p_value"].between(0, 1).all()
assert long_results.loc[~long_results["valid"], "failure_reason"].str.len().gt(0).all()
assert pd.DataFrame(full_audits)["passed"].all()
assert pd.DataFrame(full_audits)["overlap"].eq(0).all()

permutation_rows = long_results.loc[
    long_results["method"]
    == "squared_residual_feature_hsic_permutation"
].copy()
expected_permutation_rows = (
    FULL_N_REPS * 4 * 5 * len(HSIC_FEATURE_BASES)
)
assert len(permutation_rows) == expected_permutation_rows
assert permutation_rows["valid"].all()
assert permutation_rows["n_permutations"].eq(N_PERMUTATIONS).all()
assert permutation_rows["effective_rank"].eq(
    permutation_rows["n_features"]
).all()
assert permutation_rows["first_order_correction"].eq(0.0).all()
assert np.allclose(
    permutation_rows["test_statistic"],
    permutation_rows["debiased_test_statistic"],
)
assert np.allclose(
    permutation_rows["p_value"],
    permutation_rows["r_squared_permutation_p_value"],
)
assert np.allclose(
    permutation_rows["p_value"] * (N_PERMUTATIONS + 1),
    np.round(permutation_rows["p_value"] * (N_PERMUTATIONS + 1)),
)
assert permutation_rows["ess_absolute_difference"].max() <= 1e-8
assert permutation_rows["feature_orthogonality_max_abs"].max() <= 1e-10
assert permutation_rows["kernel_idempotence_frobenius"].max() <= 1e-9

# Every basis within one dataset/scheme uses the same permutation array seed.
shared_seed_counts = permutation_rows.groupby(
    [
        "repetition",
        "rho",
        "direction",
        "residual_scheme",
        "training_fraction",
    ],
    dropna=False,
)["permutation_seed"].nunique()
assert shared_seed_counts.eq(1).all()

# Analytic and permutation rows are exactly paired for every finite feature span.
pair_identity = [
    "repetition",
    "rho",
    "direction",
    "residual_scheme",
    "training_fraction",
    "feature_basis",
]
paired_variance_counts = long_results.loc[
    long_results["method"].isin(
        ["variance_score", "squared_residual_feature_hsic_permutation"]
    )
].groupby(pair_identity, dropna=False)["method"].nunique()
assert paired_variance_counts.eq(2).all()

# Expected count alone cannot detect a missing row paired with a duplicate.
result_identity = [
    "rho",
    "direction",
    "residual_scheme",
    "training_fraction",
    "method",
    "feature_basis",
    "n_bins",
    "quantile_grid",
    "repetition",
]
assert not long_results.duplicated(result_identity).any()

# Formatted tables omit q from their row labels, so explicitly verify that a
# named feature configuration has one effective rank across repetitions.
feature_rows = long_results.loc[
    long_results["valid"] & (long_results["feature_basis"] != "")
]
rank_variation = (
    feature_rows.groupby(
        [
            "rho",
            "direction",
            "scheme_label",
            "method",
            "feature_basis",
            "quantile_grid",
        ],
        dropna=False,
    )["n_features"]
    .nunique()
)
assert rank_variation.le(1).all()

# Paired-data check: one seed per (repetition, rho), shared by both directions.
seed_counts = (
    long_results.groupby(["repetition", "rho"], dropna=False)["data_seed"].nunique()
)
assert seed_counts.eq(1).all()

long_results.to_csv(LONG_CSV, index=False)
print(f"Saved {len(long_results):,} long-form rows to {LONG_CSV}.")
display(long_results.head(12))

failure_summary = (
    long_results.loc[~long_results["valid"]]
    .groupby(
        [
            "method",
            "scheme_label",
            "feature_basis",
            "n_bins",
            "quantile_grid",
            "failure_reason",
        ],
        dropna=False,
    )
    .size()
    .rename("failures")
    .reset_index()
    .sort_values("failures", ascending=False)
)
print(
    f"Valid rows: {int(long_results['valid'].sum()):,}/{len(long_results):,}; "
    f"invalid rows: {int((~long_results['valid']).sum()):,}."
)
display(failure_summary.head(30))

## Aggregate rejection tables

Each cell is `rejection_rate (MCSE); n=valid repetitions`, where

\[
{\rm MCSE}=\sqrt{\widehat p(1-\widehat p)/R}.
\]

The four data columns below are exactly the requested four principal
conditions.  Every predeclared feature, bin, quantile grid, split fraction,
and cross-fitted variant is preserved.

In [ ]:
long_results["scenario"] = [
    scenario_label(rho, direction)
    for rho, direction in zip(long_results["rho"], long_results["direction"])
]
long_results["valid_reject"] = np.where(
    long_results["valid"],
    long_results["reject"].astype(float),
    np.nan,
)

aggregate_keys = [
    "method",
    "residual_scheme",
    "scheme_label",
    "training_fraction",
    "n_folds",
    "feature_basis",
    "n_features",
    "n_bins",
    "quantile_grid",
    "scenario",
]
aggregate_numeric = (
    long_results.groupby(aggregate_keys, dropna=False, sort=False)
    .agg(
        n_valid=("valid", "sum"),
        n_total=("repetition", "size"),
        rejection_rate=("valid_reject", "mean"),
    )
    .reset_index()
)
aggregate_numeric["mcse"] = np.sqrt(
    aggregate_numeric["rejection_rate"]
    * (1.0 - aggregate_numeric["rejection_rate"])
    / aggregate_numeric["n_valid"]
)
aggregate_numeric["failures"] = (
    aggregate_numeric["n_total"] - aggregate_numeric["n_valid"]
)
aggregate_numeric.to_csv(NUMERIC_AGGREGATE_CSV, index=False)


def formatted_aggregate(
    method: str,
    row_keys: list[str],
    output_path: Path,
) -> pd.DataFrame:
    subset = aggregate_numeric.loc[aggregate_numeric["method"] == method].copy()
    subset["cell"] = subset.apply(
        lambda row: (
            f"{row['rejection_rate']:.3f} ({row['mcse']:.3f}); "
            f"n={int(row['n_valid'])}"
            if row["n_valid"] > 0
            else f"invalid; n=0/{int(row['n_total'])}"
        ),
        axis=1,
    )
    wide = subset.pivot_table(
        index=row_keys,
        columns="scenario",
        values="cell",
        aggfunc="first",
        dropna=False,
        observed=True,
    )
    wide = wide.reindex(columns=SCENARIO_ORDER)
    wide.to_csv(output_path)
    return wide


variance_table = formatted_aggregate(
    "variance_score",
    ["scheme_label", "feature_basis"],
    VARIANCE_CSV,
)
permutation_hsic_table = formatted_aggregate(
    "squared_residual_feature_hsic_permutation",
    ["scheme_label", "feature_basis"],
    PERMUTATION_HSIC_CSV,
)
levene_table = formatted_aggregate(
    "levene",
    ["scheme_label", "n_bins"],
    LEVENE_CSV,
)
brown_forsythe_table = formatted_aggregate(
    "brown_forsythe",
    ["scheme_label", "n_bins"],
    BROWN_FORSYTHE_CSV,
)
quantile_table = formatted_aggregate(
    "quantile_regression",
    ["scheme_label", "quantile_grid", "feature_basis"],
    QUANTILE_CSV,
)

display(Markdown("### Variance-score tests: analytic chi-square calibration"))
display(variance_table)
display(Markdown("### Squared-residual finite-feature HSIC: permutation calibration"))
display(permutation_hsic_table)
display(Markdown("### Levene tests"))
display(levene_table)
display(Markdown("### Brown--Forsythe tests"))
display(brown_forsythe_table)
display(Markdown("### Quantile-regression tests"))
display(quantile_table)

## Analytic versus permutation calibration of the same statistic

The first 70 rows below are deliberately paired: for every residual scheme
and feature basis, the analytic BP/White-style result appears immediately
beside its projection-kernel permutation-HSIC counterpart. Both use the same
squared residual, feature span, effective rank, and fitted first stage; only
the null calibration differs; their statistics are fixed monotone transforms
of the same explained-sum-of-squares signal. Each cell is
`rejection rate (MCSE); n=valid`.

The final two rows reuse the saved paper results as references. `cross-fit`
is the paper's non-debiased raw-residual Gaussian-kernel permutation HSIC;
`debiased` is its reverse-triangle bootstrap p-value. They are omnibus
raw-residual tests and do not target only the finite squared-residual moments.

In [ ]:
FINITE_FEATURE_METHOD_LABELS = {
    "variance_score": "analytic BP/White",
    "squared_residual_feature_hsic_permutation": "permutation finite-feature HSIC",
}
SCHEME_ORDER = {
    "cross-fit (5 folds)": 0,
    "split 20% train": 1,
    "split 40% train": 2,
    "split 60% train": 3,
    "split 80% train": 4,
}
BASIS_ORDER = {basis: index for index, basis in enumerate(VARIANCE_BASES)}
CALIBRATION_ORDER = {
    "analytic BP/White": 0,
    "permutation finite-feature HSIC": 1,
    "cross-fit": 0,
    "debiased": 1,
}

finite_feature_comparison = aggregate_numeric.loc[
    aggregate_numeric["method"].isin(FINITE_FEATURE_METHOD_LABELS)
].copy()
finite_feature_comparison["calibration"] = finite_feature_comparison[
    "method"
].map(FINITE_FEATURE_METHOD_LABELS)
finite_feature_comparison["test_family"] = "targeted squared-residual feature test"
finite_feature_comparison["scheme_order"] = finite_feature_comparison[
    "scheme_label"
].map(SCHEME_ORDER)
finite_feature_comparison["basis_order"] = finite_feature_comparison[
    "feature_basis"
].map(BASIS_ORDER)
finite_feature_comparison["calibration_order"] = finite_feature_comparison[
    "calibration"
].map(CALIBRATION_ORDER)

paper_records = pd.read_csv(BASELINE_REPETITIONS_CSV)
paper_reference_rows = []
for (rho, report_direction), scenario in zip(
    [(0.0, "X->Y"), (0.0, "Y->X"), (0.35, "X->Y"), (0.35, "Y->X")],
    SCENARIO_ORDER,
):
    repo_direction = {"X->Y": "Y on X", "Y->X": "X on Y"}[
        report_direction
    ]
    scenario_records = paper_records.loc[
        np.isclose(paper_records["rho"], rho)
        & (paper_records["direction"] == repo_direction)
    ]
    for calibration, method, p_value_column in (
        ("cross-fit", "core full naive", "p_value"),
        ("debiased", "core full debiased", "bootstrap_p_value"),
    ):
        values = scenario_records.loc[
            scenario_records["method"] == method, p_value_column
        ].dropna().to_numpy(dtype=float)
        # Preserve the paper notebook's existing <= 0.05 convention.
        reject = values <= ALPHA
        rejection_rate = float(np.mean(reject))
        paper_reference_rows.append(
            {
                "method": method,
                "test_family": "paper raw-residual HSIC reference",
                "scheme_label": "paper raw-residual HSIC",
                "training_fraction": 0.8,
                "n_folds": N_FOLDS,
                "feature_basis": "Gaussian predictor/residual kernels",
                "n_features": np.nan,
                "scenario": scenario,
                "calibration": calibration,
                "n_valid": len(values),
                "n_total": len(values),
                "rejection_rate": rejection_rate,
                "mcse": math.sqrt(
                    rejection_rate * (1.0 - rejection_rate) / len(values)
                ),
                "failures": 0,
                "scheme_order": 5,
                "basis_order": 0,
                "calibration_order": CALIBRATION_ORDER[calibration],
            }
        )
paper_reference_numeric = pd.DataFrame(paper_reference_rows)

calibration_comparison_numeric = pd.concat(
    [finite_feature_comparison, paper_reference_numeric],
    ignore_index=True,
    sort=False,
)
calibration_comparison_numeric["cell"] = calibration_comparison_numeric.apply(
    lambda row: (
        f"{row['rejection_rate']:.3f} ({row['mcse']:.3f}); "
        f"n={int(row['n_valid'])}"
    ),
    axis=1,
)
calibration_comparison_numeric = calibration_comparison_numeric.sort_values(
    ["scheme_order", "basis_order", "calibration_order", "scenario"]
)
calibration_comparison_table = calibration_comparison_numeric.pivot_table(
    index=["scheme_label", "feature_basis", "calibration"],
    columns="scenario",
    values="cell",
    aggfunc="first",
    observed=True,
)
ordered_index = (
    calibration_comparison_numeric[
        [
            "scheme_label",
            "feature_basis",
            "calibration",
            "scheme_order",
            "basis_order",
            "calibration_order",
        ]
    ]
    .drop_duplicates()
    .sort_values(["scheme_order", "basis_order", "calibration_order"])
    .set_index(["scheme_label", "feature_basis", "calibration"])
    .index
)
calibration_comparison_table = calibration_comparison_table.reindex(
    index=ordered_index,
    columns=SCENARIO_ORDER,
)
assert calibration_comparison_table.shape == (72, 4)
assert calibration_comparison_table.notna().all().all()
calibration_comparison_table.to_csv(CALIBRATION_COMPARISON_CSV)
display(calibration_comparison_table)
print(f"Saved paired calibration comparison to {CALIBRATION_COMPARISON_CSV}.")

## Empirical interpretation

The text below is generated from the complete aggregate table.  Summaries
average only to describe broad sensitivity; all individual configurations
remain visible above.  No test is calibrated from these null simulations and
no configuration is removed after observing its result.

In [ ]:
def numeric_slice(method: str | None = None) -> pd.DataFrame:
    frame = aggregate_numeric.copy()
    if method is not None:
        frame = frame.loc[frame["method"] == method]
    return frame


def rate_for(
    method: str,
    scenario: str,
    scheme_label: str | None = None,
    feature_basis: str | None = None,
    n_bins: int | None = None,
    quantile_grid: str | None = None,
) -> float:
    frame = aggregate_numeric.loc[
        (aggregate_numeric["method"] == method)
        & (aggregate_numeric["scenario"] == scenario)
    ]
    for column, value in (
        ("scheme_label", scheme_label),
        ("feature_basis", feature_basis),
        ("n_bins", n_bins),
        ("quantile_grid", quantile_grid),
    ):
        if value is not None:
            frame = frame.loc[frame[column] == value]
    if len(frame) != 1:
        return np.nan
    return float(frame.iloc[0]["rejection_rate"])


null_panel = "rho=0, X->Y (size)"
power_panels = SCENARIO_ORDER[1:]

size_by_method_scheme = (
    aggregate_numeric.loc[aggregate_numeric["scenario"] == null_panel]
    .groupby(["method", "scheme_label"], observed=True)["rejection_rate"]
    .agg(["min", "median", "max"])
)
display(Markdown("### Null rejection ranges across hyperparameters"))
display(size_by_method_scheme)

# Split-fraction trend: median across every configuration in each family.
split_size_trend = (
    aggregate_numeric.loc[
        (aggregate_numeric["scenario"] == null_panel)
        & (aggregate_numeric["residual_scheme"] == "sample_split")
    ]
    .groupby(["method", "training_fraction"], observed=True)["rejection_rate"]
    .median()
    .unstack("training_fraction")
)
display(Markdown("### Median null rejection by split training fraction"))
display(split_size_trend)

# Variance bases: average power over all three alternatives and all schemes.
variance_rank = (
    aggregate_numeric.loc[
        (aggregate_numeric["method"] == "variance_score")
        & (aggregate_numeric["scenario"].isin(power_panels))
    ]
    .groupby("feature_basis", observed=True)["rejection_rate"]
    .mean()
    .sort_values(ascending=False)
)
display(Markdown("### Variance-basis mean power across three alternative panels and all schemes"))
display(variance_rank.to_frame("mean rejection"))

# Method-level cross-fit vs split summary.
scheme_comparison = (
    aggregate_numeric.assign(
        scheme_class=np.where(
            aggregate_numeric["residual_scheme"] == "cross_fitted_variant",
            "cross-fitted variant",
            "ordinary splits",
        ),
        panel_role=np.where(
            aggregate_numeric["scenario"] == null_panel,
            "size",
            "power",
        ),
    )
    .groupby(["method", "scheme_class", "panel_role"], observed=True)[
        "rejection_rate"
    ]
    .median()
    .unstack(["scheme_class", "panel_role"])
)
display(Markdown("### Median rejection across configurations: cross-fit versus splits"))
display(scheme_comparison)

inflated = aggregate_numeric.loc[
    (aggregate_numeric["scenario"] == null_panel)
    & (aggregate_numeric["rejection_rate"] > 0.10)
]
inflated_by_method = inflated.groupby("method", observed=True).size().to_dict()

best_variance_basis = str(variance_rank.index[0])
fourier2_hetero_power = rate_for(
    "variance_score",
    "rho=0.35, X->Y (power)",
    scheme_label="cross-fit (5 folds)",
    feature_basis="fourier2",
)
poly2_hetero_power = rate_for(
    "variance_score",
    "rho=0.35, X->Y (power)",
    scheme_label="cross-fit (5 folds)",
    feature_basis="poly2",
)
fourier2_60_power = rate_for(
    "variance_score",
    "rho=0.35, X->Y (power)",
    scheme_label="split 60% train",
    feature_basis="fourier2",
)
poly2_60_power = rate_for(
    "variance_score",
    "rho=0.35, X->Y (power)",
    scheme_label="split 60% train",
    feature_basis="poly2",
)

levene_by_bins = (
    aggregate_numeric.loc[aggregate_numeric["method"] == "levene"]
    .groupby("n_bins", observed=True)["rejection_rate"]
    .agg(["min", "max"])
)
bf_by_bins = (
    aggregate_numeric.loc[aggregate_numeric["method"] == "brown_forsythe"]
    .groupby("n_bins", observed=True)["rejection_rate"]
    .agg(["min", "max"])
)
quantile_by_grid_basis = (
    aggregate_numeric.loc[aggregate_numeric["method"] == "quantile_regression"]
    .groupby(["quantile_grid", "feature_basis"], observed=True)["rejection_rate"]
    .agg(["min", "max"])
)

invalid_count = int((~long_results["valid"]).sum())
invalid_quantile_count = int(
    ((long_results["method"] == "quantile_regression") & ~long_results["valid"]).sum()
)


def crossfit_size_range(method: str) -> tuple[float, float]:
    row = size_by_method_scheme.loc[(method, "cross-fit (5 folds)")]
    return float(row["min"]), float(row["max"])


def median_scheme_rate(method: str, scheme_class: str, role: str) -> float:
    return float(scheme_comparison.loc[method, (scheme_class, role)])


def split_trend(method: str) -> str:
    row = split_size_trend.loc[method]
    return "/".join(f"{row[fraction]:.3f}" for fraction in (0.2, 0.4, 0.6, 0.8))


levene_cf_min, levene_cf_max = crossfit_size_range("levene")
bf_cf_min, bf_cf_max = crossfit_size_range("brown_forsythe")
variance_cf_min, variance_cf_max = crossfit_size_range("variance_score")
quantile_cf_min, quantile_cf_max = crossfit_size_range("quantile_regression")

direct_q_fourier4_wide = float(
    null_smoke.loc[
        (null_smoke["method"] == "quantile_regression")
        & (null_smoke["configuration"] == "fourier4, wide"),
        "rejection_rate",
    ].iloc[0]
)

qr_cf_central = aggregate_numeric.loc[
    (aggregate_numeric["method"] == "quantile_regression")
    & (aggregate_numeric["scenario"] == null_panel)
    & (aggregate_numeric["scheme_label"] == "cross-fit (5 folds)")
    & (aggregate_numeric["quantile_grid"] == "central"),
    "rejection_rate",
]
qr_cf_wide = aggregate_numeric.loc[
    (aggregate_numeric["method"] == "quantile_regression")
    & (aggregate_numeric["scenario"] == null_panel)
    & (aggregate_numeric["scheme_label"] == "cross-fit (5 folds)")
    & (aggregate_numeric["quantile_grid"] == "wide"),
    "rejection_rate",
]

interpretation = f"""
### Concise findings

- **Which procedures are calibrated?** The clearest cross-fitted calibration
  is for Levene (size `{levene_cf_min:.3f}`--`{levene_cf_max:.3f}` over B) and
  Brown--Forsythe (`{bf_cf_min:.3f}`--`{bf_cf_max:.3f}`, mildly conservative
  for larger B). Cross-fitted variance scores are mildly liberal except for
  `linear`: their range is `{variance_cf_min:.3f}`--`{variance_cf_max:.3f}`
  (`linear=0.060`; most richer bases are 0.090--0.110). All seven variance
  bases are well calibrated with an 80% split (0.030--0.060), at the cost of
  only 50 evaluation observations. Cross-fitted central quantile tests reject
  exactly `{qr_cf_central.min():.3f}`--`{qr_cf_central.max():.3f}` and are
  severely conservative; wide-grid sizes are `{qr_cf_wide.min():.3f}`--
  `{qr_cf_wide.max():.3f}`, with `fourier4` at 0.100 rather than reliably
  calibrated.
- **Training fraction.** Median null rejection at 20/40/60/80% training is
  `{split_trend('variance_score')}` for variance scores,
  `{split_trend('levene')}` for Levene,
  `{split_trend('brown_forsythe')}` for Brown--Forsythe, and
  `{split_trend('quantile_regression')}` for quantile regression. More
  first-stage training sharply reduces residual-estimation distortion for
  variance and quantile tests. Levene is non-monotone at 80% because small
  evaluation bins degrade its F approximation; Brown--Forsythe becomes
  conservative there.
- **Does cross-fitting help?** Median cross-fitted size/power is
  `{median_scheme_rate('variance_score', 'cross-fitted variant', 'size'):.3f}`/
  `{median_scheme_rate('variance_score', 'cross-fitted variant', 'power'):.3f}`
  for variance scores,
  `{median_scheme_rate('levene', 'cross-fitted variant', 'size'):.3f}`/
  `{median_scheme_rate('levene', 'cross-fitted variant', 'power'):.3f}` for
  Levene, and
  `{median_scheme_rate('brown_forsythe', 'cross-fitted variant', 'size'):.3f}`/
  `{median_scheme_rate('brown_forsythe', 'cross-fitted variant', 'power'):.3f}`
  for Brown--Forsythe. The corresponding ordinary-split median powers are
  `{median_scheme_rate('variance_score', 'ordinary splits', 'power'):.3f}`,
  `{median_scheme_rate('levene', 'ordinary splits', 'power'):.3f}`, and
  `{median_scheme_rate('brown_forsythe', 'ordinary splits', 'power'):.3f}`.
  Thus cross-fitting gives the best calibration/power balance for dispersion
  tests and restores evaluation sample size relative to the calibrated 80%
  splits. Quantile splitting appears more powerful (median 0.465 versus 0.270)
  only alongside median size 0.300 versus 0.005, so that comparison is not
  evidence of genuine power. Cross-fitting remains a diagnostic variant, not
  a proof of its nominal reference law.
- **Variance bases.** Averaged over all three alternatives and schemes, the
  leading raw-power basis is `{best_variance_basis}`, but raw rankings mix
  power and size. In the heteroskedastic forward panel, cross-fitted
  `fourier2` and `poly2` reject `{fourier2_hetero_power:.3f}` and
  `{poly2_hetero_power:.3f}` with similar null sizes 0.110 and 0.100. At the
  60% split, `fourier2` improves power from `{poly2_60_power:.3f}` to
  `{fourier2_60_power:.3f}` at similar size (0.090 versus 0.100), evidence that
  sinusoidal features help. `fourier8` loses power when the evaluation sample
  is small because its 16 degrees of freedom are costly. These Fourier tests
  are targeted, not oracle: frequencies were fixed without realized DGP
  coefficients or phases.
- **Binning.** Cross-fitted Levene and Brown--Forsythe both have power 1.000
  for every B against forward heteroskedasticity. More bins help Levene on the
  reverse alternatives but can increase size; for the 80% split, Levene B=10
  has size 0.170. Median-centering is more conservative: the same split's
  Brown--Forsythe B=10 has size 0.000 and forward-heterogeneity power only
  0.020. Hence B sensitivity is material, especially with only five residuals
  per bin.
- **Quantiles and numerical stability.** Cross-fitted central-grid forward
  heterogeneity power is only 0.230--0.530, while the wide grid gives
  0.700--0.990. The latter is not uniformly trustworthy: direct observed-error
  smoke size for wide `fourier4` is `{direct_q_fourier4_wide:.3f}`, and its
  feasible cross-fitted size is 0.100. Sample-split quantile size remains
  grossly inflated even at 80% training (median 0.151), so its high apparent
  power is largely first-stage distortion. There were
  `{invalid_quantile_count}` invalid global quantile rows (`{invalid_count}`
  rows overall), all due to the predeclared 10,000-iteration limit and all
  retained with their failure reasons.
- **Size-qualified power.** In total, `{len(inflated)}` aggregate
  configurations have null rejection above 0.10: variance
  `{inflated_by_method.get('variance_score', 0)}`, Levene
  `{inflated_by_method.get('levene', 0)}`, Brown--Forsythe
  `{inflated_by_method.get('brown_forsythe', 0)}`, quantile
  `{inflated_by_method.get('quantile_regression', 0)}`, and squared-residual
  permutation HSIC
  `{inflated_by_method.get('squared_residual_feature_hsic_permutation', 0)}`.
  Their alternative
  rejection rates cannot be read as power without the adjacent size column.
- **A defensible paper subset must be fixed by design, not selected from the
  rankings above.** A small a-priori set is: `poly2` as the standard 1D White
  variance score; `fourier2` as the declared smooth-periodic targeted score;
  both Levene and Brown--Forsythe with B=4 as moderate-bin mean/median checks;
  and central-grid `poly2` quantile regression. A balanced 60% split and the
  five-fold cross-fitted variant are design-based reporting choices, not
  winners selected from this table; their observed miscalibration must remain
  visible, with the complete sensitivity tables retained.
- **Scope.** These procedures test finite variance, binned dispersion, or
  finite quantile restrictions. None is an omnibus residual-independence test
  like characteristic-kernel HSIC.
"""
display(Markdown(interpretation))

## Interpretation of permutation calibration

The summaries below compare calibration methods only after matching the
feature basis and residual scheme. Raw rejection probabilities are never
called power without the adjacent true-null rejection rate.

In [ ]:
PERMUTATION_METHOD = "squared_residual_feature_hsic_permutation"

analytic_null = aggregate_numeric.loc[
    (aggregate_numeric["method"] == "variance_score")
    & (aggregate_numeric["scenario"] == null_panel),
    ["scheme_label", "training_fraction", "feature_basis", "rejection_rate"],
].rename(columns={"rejection_rate": "analytic_size"})
permutation_null = aggregate_numeric.loc[
    (aggregate_numeric["method"] == PERMUTATION_METHOD)
    & (aggregate_numeric["scenario"] == null_panel),
    ["scheme_label", "training_fraction", "feature_basis", "rejection_rate"],
].rename(columns={"rejection_rate": "permutation_size"})
paired_null_calibration = analytic_null.merge(
    permutation_null,
    on=["scheme_label", "training_fraction", "feature_basis"],
    validate="one_to_one",
)
paired_null_calibration["permutation_minus_analytic"] = (
    paired_null_calibration["permutation_size"]
    - paired_null_calibration["analytic_size"]
)
display(Markdown("### Matched true-null calibration"))
display(paired_null_calibration)

permutation_split_size = (
    permutation_null.groupby("training_fraction", observed=True)[
        "permutation_size"
    ]
    .agg(["min", "median", "max"])
)
permutation_cf_size = permutation_null.loc[
    permutation_null["scheme_label"] == "cross-fit (5 folds)",
    "permutation_size",
]
display(Markdown("### Permutation-HSIC null size by first-stage scheme"))
display(permutation_split_size)

permutation_cf_power = aggregate_numeric.loc[
    (aggregate_numeric["method"] == PERMUTATION_METHOD)
    & (aggregate_numeric["scheme_label"] == "cross-fit (5 folds)")
    & aggregate_numeric["scenario"].isin(power_panels),
    ["feature_basis", "scenario", "rejection_rate"],
].pivot(index="feature_basis", columns="scenario", values="rejection_rate")
permutation_cf_power = permutation_cf_power.reindex(VARIANCE_BASES)
display(Markdown("### Cross-fitted permutation-HSIC rejection under alternatives"))
display(permutation_cf_power)

heterogeneous_forward = "rho=0.35, X->Y (power)"
fourier_forward = permutation_cf_power.loc[
    ["fourier1", "fourier2", "fourier4", "fourier8"],
    heterogeneous_forward,
]
permutation_inflated = permutation_null.loc[
    permutation_null["permutation_size"] > 0.10
]
max_abs_calibration_change = float(
    paired_null_calibration["permutation_minus_analytic"].abs().max()
)
median_abs_calibration_change = float(
    paired_null_calibration["permutation_minus_analytic"].abs().median()
)

paper_rate_lookup = paper_reference_numeric.set_index(
    ["calibration", "scenario"]
)["rejection_rate"]

permutation_findings = f"""
### Findings specific to the new calibration

1. **Analytic versus permutation size for the identical statistic.** Across
   the 35 matched basis/scheme rows, changing only calibration changes null
   rejection by a median absolute `{median_abs_calibration_change:.3f}` and a
   maximum `{max_abs_calibration_change:.3f}`. Thus the two calibrations are
   usually very similar. Equivalence up to a fixed monotone scaling is
   asserted numerically, so the remaining differences come only from the
   analytic chi-square versus empirical permutation reference distribution.
2. **Small first-stage training splits.** With 20/40/60/80% training, median
   permutation-HSIC null rejection is
   `{permutation_split_size.loc[0.2, 'median']:.3f}`/
   `{permutation_split_size.loc[0.4, 'median']:.3f}`/
   `{permutation_split_size.loc[0.6, 'median']:.3f}`/
   `{permutation_split_size.loc[0.8, 'median']:.3f}`. Thus permutation does
   not erase predictor-dependent error left by a poorly estimated first
   stage; the trend must be read as feasible-residual calibration, not as an
   oracle property.
3. **Cross-fitting versus splitting.** Cross-fitted permutation sizes range
   from `{permutation_cf_size.min():.3f}` to
   `{permutation_cf_size.max():.3f}` (median
   `{permutation_cf_size.median():.3f}`). It is much better calibrated than
   the 20% and 40% training splits, roughly comparable to the 60% split, but
   **not** better calibrated than the 80% split (median
   `{permutation_split_size.loc[0.8, 'median']:.3f}`). Cross-fitting evaluates
   all 250 observations and can increase power, but its observed null size—not
   cross-fitting alone—determines whether that power is interpretable.
4. **Fourier sensitivity.** Against the sinusoidal forward scale alternative,
   cross-fitted Fourier-1/2/4/8 rejection is
   `{fourier_forward['fourier1']:.3f}`/
   `{fourier_forward['fourier2']:.3f}`/
   `{fourier_forward['fourier4']:.3f}`/
   `{fourier_forward['fourier8']:.3f}`; linear is already
   `{permutation_cf_power.loc['linear', heterogeneous_forward]:.3f}`, so that
   panel is near a ceiling. Fourier features matter more in the reverse
   panels: rejection rises from
   `{permutation_cf_power.loc['linear', power_panels[0]]:.3f}` to
   `{permutation_cf_power.loc['fourier4', power_panels[0]]:.3f}` at rho=0 and
   from `{permutation_cf_power.loc['linear', power_panels[2]]:.3f}` to
   `{permutation_cf_power.loc['fourier4', power_panels[2]]:.3f}` at rho=.35,
   though the adjacent null sizes must still qualify those gains.
5. **Cost of extra frequencies.** Effective rank rises from 2 to 4, 8, and
   16 for Fourier-1/2/4/8. Fourier-8 falls to
   `{fourier_forward['fourier8']:.3f}` in the forward scale panel and to
   `{permutation_cf_power.loc['fourier8', power_panels[0]]:.3f}` versus
   Fourier-4's `{permutation_cf_power.loc['fourier4', power_panels[0]]:.3f}`
   in the rho=0 reverse panel. This modest loss is consistent with extra noise
   directions diluting finite-sample signal.
6. **Size-qualified power.** `{len(permutation_inflated)}` of the 35
   permutation configurations have true-null rejection above 0.10. Any high
   alternative rejection in those matched rows is flagged as size-inflated
   evidence rather than clean power.
7. **Targeted versus omnibus paper HSIC.** The paper raw-residual `cross-fit`
   reference has size
   `{paper_rate_lookup.loc[('cross-fit', null_panel)]:.3f}` and alternative
   rejection
   `{paper_rate_lookup.loc[('cross-fit', power_panels[0])]:.3f}`/
   `{paper_rate_lookup.loc[('cross-fit', power_panels[1])]:.3f}`/
   `{paper_rate_lookup.loc[('cross-fit', power_panels[2])]:.3f}`. The paper
   `debiased` reference is
   `{paper_rate_lookup.loc[('debiased', null_panel)]:.3f}` and
   `{paper_rate_lookup.loc[('debiased', power_panels[0])]:.3f}`/
   `{paper_rate_lookup.loc[('debiased', power_panels[1])]:.3f}`/
   `{paper_rate_lookup.loc[('debiased', power_panels[2])]:.3f}`. Among the
   close-to-sized cross-fitted targeted rows (null rejection at most .10),
   the corresponding best alternative rates are
   `{permutation_cf_power.loc[['linear', 'fourier4', 'fourier8'], power_panels[0]].max():.3f}`/
   `{permutation_cf_power.loc[['linear', 'fourier4', 'fourier8'], power_panels[1]].max():.3f}`/
   `{permutation_cf_power.loc[['linear', 'fourier4', 'fourier8'], power_panels[2]].max():.3f}`.
   Thus the targeted test matches the forward-scale ceiling but does not beat
   the debiased omnibus method in either reverse panel. The Gaussian-kernel
   references target general predictor/residual dependence; the new rows
   target only covariance with squared residuals in the declared feature span.
"""
display(Markdown(permutation_findings))

### Statistical scope and validity

- With oracle residuals, permutation calibration is finite-sample exact under
  predictor--residual independence. The separate oracle experiment above
  gives approximately uniform p-values and near-5% rejection for all seven
  bases.
- With estimated split or cross-fitted residuals, permutation is **not exact**:
  the residuals retain predictor-dependent first-stage estimation error and
  are not an exchangeable oracle noise sample.
- For the scalar squared-residual feature, the leading mean-regression error
  cancels because \(E[2\varepsilon\mid W]=0\). The first-order debiased and
  plug-in statistics therefore coincide, leaving a second-order first-stage
  remainder that should shrink as the mean regression becomes accurate.
- Cross-fitting uses more evaluation observations and should generally
  improve power, but it does not itself establish permutation validity.
- The finite-feature procedure detects only variance patterns represented in
  the fixed span of `linear`, polynomial, or declared Fourier features. It is
  not an omnibus residual-independence test.

## Artifact manifest and reproducibility metadata

In [ ]:
failure_reason_counts = (
    long_results.loc[~long_results["valid"], "failure_reason"]
    .value_counts()
    .to_dict()
)
metadata = {
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_notebook": str(SOURCE_NOTEBOOK),
    "source_dgp_cell_id": DGP_CELL_ID,
    "source_dgp_sha256": DGP_SHA256,
    "source_config_cell_id": CONFIG_CELL_ID,
    "source_config_sha256": CONFIG_SHA256,
    "baseline_run_json": str(BASELINE_RUN_JSON),
    "config": {
        "sample_size": CORE_N,
        "repetitions_run": FULL_N_REPS,
        "authoritative_repetitions": CORE_N_REPS,
        "rho_values": list(RHO_VALUES),
        "directions": list(DIRECTION_MAP.values()),
        "alpha": ALPHA,
        "data_seed_base": DATA_SEED_BASE,
        "data_seed_formula": "82345 + 1000*repetition + 100*rho_index",
        "n_folds": N_FOLDS,
        "split_training_fractions": [p / N_FOLDS for p in SPLIT_PREFIXES],
        "variance_bases": list(VARIANCE_BASES),
        "squared_residual_feature_hsic_bases": list(HSIC_FEATURE_BASES),
        "squared_residual_feature_hsic_permutations": N_PERMUTATIONS,
        "squared_residual_feature_hsic_permutation_seed_base": PERMUTATION_SEED_BASE,
        "squared_residual_feature_hsic_statistic_normalization": HSIC_STATISTIC_NORMALIZATION,
        "squared_residual_feature_hsic_rejection_rule": "p < 0.05",
        "squared_residual_feature_hsic_first_order_correction": 0.0,
        "squared_residual_feature_hsic_feasible_calibration": (
            "plug-in permutation; heuristic for estimated residuals"
        ),
        "bin_counts": list(BIN_COUNTS),
        "quantile_bases": list(QUANTILE_BASES),
        "quantile_grids": {key: list(value) for key, value in QUANTILE_GRIDS.items()},
        "minimum_bin_size": MIN_BIN_SIZE,
        "quantreg_max_iter": QUANTREG_MAX_ITER,
        "quantreg_p_tol": QUANTREG_P_TOL,
    },
    "first_stage": {
        "estimator": "sklearn KernelRidge with Gaussian/RBF kernel",
        "cv_folds": CORE_CONFIG.mean_cv_folds,
        "alphas": list(CORE_CONFIG.mean_alphas),
        "length_scales": list(CORE_CONFIG.mean_length_scales),
        "scoring": "neg_mean_squared_error",
        "inner_and_outer_shuffle": False,
        "permutation_hsic_refits_first_stage": False,
        "permutation_hsic_reuses_common_residual_arrays": True,
    },
    "oracle_permutation_sanity": {
        "rho": 0.0,
        "direction": "X->Y",
        "residual": "true structural noise",
        "repetitions": ORACLE_PERMUTATION_REPS,
        "sample_size": CORE_N,
        "rejection_rate_min": float(
            oracle_permutation_summary["rejection_rate"].min()
        ),
        "rejection_rate_max": float(
            oracle_permutation_summary["rejection_rate"].max()
        ),
        "minimum_uniform_ks_p_value": float(
            oracle_permutation_summary["uniform_ks_p_value"].min()
        ),
    },
    "runtime_seconds": full_runtime_seconds,
    "row_count": len(long_results),
    "valid_rows": int(long_results["valid"].sum()),
    "invalid_rows": int((~long_results["valid"]).sum()),
    "failure_reason_counts": failure_reason_counts,
    "verification": {
        "all_checks_passed": bool(verification["passed"].all()),
        "baseline_residual_max_absolute_difference": max_residual_difference,
        "baseline_residual_exact_match": exact_residual_match,
        "direct_error_null_rejection_min": float(null_smoke["rejection_rate"].min()),
        "direct_error_null_rejection_max": float(null_smoke["rejection_rate"].max()),
        "strong_variance_fourier2_power": strong_power,
        "strong_variance_fourier2_mcse": strong_mcse,
        "max_manual_statsmodels_wald_p_difference": max_wald_difference,
        "projection_hsic_all_checks_passed": bool(
            all(item["passed"] for item in projection_check_rows)
        ),
        "projection_kernel_max_symmetry_error": float(
            max(item["kernel_symmetry_max_abs"] for item in projection_check_rows)
        ),
        "projection_kernel_max_idempotence_error": float(
            max(item["kernel_idempotence_max_abs"] for item in projection_check_rows)
        ),
        "projection_hsic_max_ess_difference": float(
            max(item["hsic_vs_ols_ess_abs"] for item in projection_check_rows)
        ),
    },
    "versions": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "scipy": scipy.__version__,
        "pandas": pd.__version__,
        "sklearn": sklearn.__version__,
        "statsmodels": statsmodels.__version__,
    },
    "artifacts": [
        "toy_additive_noise_classical_diagnostics.ipynb",
        str(LONG_CSV),
        str(VARIANCE_CSV),
        str(PERMUTATION_HSIC_CSV),
        str(CALIBRATION_COMPARISON_CSV),
        str(ORACLE_PERMUTATION_LONG_CSV),
        str(ORACLE_PERMUTATION_SUMMARY_CSV),
        str(LEVENE_CSV),
        str(BROWN_FORSYTHE_CSV),
        str(QUANTILE_CSV),
        str(NUMERIC_AGGREGATE_CSV),
        str(VERIFICATION_CSV),
        str(NULL_SMOKE_CSV),
        str(POWER_SMOKE_CSV),
        str(PROGRESS_JSON),
        str(METADATA_JSON),
    ],
}
with METADATA_JSON.open("w", encoding="utf-8") as handle:
    json.dump(metadata, handle, indent=2)

artifact_manifest = pd.DataFrame(
    {
        "artifact": [
            "executed notebook (this file)",
            "long-form results",
            "variance-score aggregate",
            "squared-residual permutation-HSIC aggregate",
            "paired analytic/permutation calibration comparison",
            "oracle permutation-HSIC long form",
            "oracle permutation-HSIC summary",
            "Levene aggregate",
            "Brown--Forsythe aggregate",
            "quantile aggregate",
            "numeric aggregate",
            "verification checks",
            "direct-error null smoke",
            "strong-variance smoke",
            "metadata",
            "progress/status",
        ],
        "path": [
            "toy_additive_noise_classical_diagnostics.ipynb",
            LONG_CSV,
            VARIANCE_CSV,
            PERMUTATION_HSIC_CSV,
            CALIBRATION_COMPARISON_CSV,
            ORACLE_PERMUTATION_LONG_CSV,
            ORACLE_PERMUTATION_SUMMARY_CSV,
            LEVENE_CSV,
            BROWN_FORSYTHE_CSV,
            QUANTILE_CSV,
            NUMERIC_AGGREGATE_CSV,
            VERIFICATION_CSV,
            NULL_SMOKE_CSV,
            POWER_SMOKE_CSV,
            METADATA_JSON,
            PROGRESS_JSON,
        ],
    }
)
display(artifact_manifest)
print("Analysis complete.")